# Answer-flip study — main run, LLaVA-OneVision-7B (Run-All)

One notebook for all five benchmarks. Each run handles **one** benchmark, chosen in the config cell. Everything it does follows the pre-registration (sampling seeds, methods, budgets, four noise passes).

| `BENCHMARK` | Questions | Expected time | Videos downloaded |
|---|---|---|---|
| `"videomme"` | about 810 (chunks 02, 08, 16, 18, 20) | 6–7 h | 5 × 5 GB zips |
| `"mvbench"` | about 475 (25 per task, 19 tasks) | 4–6 h | one small file per video |
| `"egoschema"` | 500 | 6–7 h | 5 × 21 GB zips (only the needed videos are kept) |
| `"longvideobench"` | 500 (125 per duration group) | 6–8 h | 31 × 5 GB parts, streamed one at a time |
| `"mlvu"` | 504 (72 per task, 7 tasks) | 6–8 h | one file per video |

**What happens per video:** 3 frame sets are decoded (original, and shifted by ½ and ¼ of a sampling interval), the 23 compressed versions are built once, and the system prompt + video tokens are run through the model **once per version**. Every question and option order then reuses that cached prefix. FlashVID and HoliTom act inside the model and depend on the question, so they run in full per question.

**Two safety checks stop the run before wasting GPU hours:** (1) our pipeline vs the official model forward, (2) cached vs uncached answers on the first video.

## How to run
1. Settings → Accelerator **GPU T4 x2**, Internet **On**, HF_TOKEN in Add-ons → Secrets (already set; MVBench, LongVideoBench and MLVU terms accepted).
2. First time only, a free CPU check of all five datasets: `DATA_CHECK_ONLY = True`, Accelerator **None** → **Run All** (~30–40 min, mostly one 21 GB EgoSchema download). It stops after cell 3 with "Data check finished" — that stop is expected. Send me `data_check.json`.
3. Then `DATA_CHECK_ONLY = False`, Accelerator GPU, `BENCHMARK = "videomme"`, `QUICK_TEST = True` → **Run All** (~25 min). Check: `Compressor self-test OK ✅`, equivalence check `PASS`, `Cache check: PASS`, a few ✓ lines.
4. Then `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**, one benchmark per run.
5. If a run stops at the 12-hour limit: open the next version, **Add Input** → the previous version's output, and run again. It continues where it stopped.
6. Send back from `flips_main_<benchmark>/`: `summary_<benchmark>.json`, `cells.csv`, `summary_<benchmark>.png`, `results.jsonl`, `cache_check.json`, `equivalence_check.json`, `manifest_<benchmark>.json`, `run_record_*.json` (+ `errors.log` if not empty).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CONFIG — change only BENCHMARK and QUICK_TEST                    ║
# ╚══════════════════════════════════════════════════════════════════╝
BENCHMARK  = "mlvu"       # one of: "videomme", "mvbench", "egoschema", "longvideobench", "mlvu"
QUICK_TEST = False        # True = ~25-min smoke test (tiny model, a few questions). False = the real run.
DATA_CHECK_ONLY = False   # True = CPU-only check of all five benchmarks' data (no GPU needed); stops after cell 3.

MODEL_ID_REAL  = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
MODEL_ID_QUICK = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
SIGLIP_ID      = "google/siglip-so400m-patch14-384"   # FastVID uses the original SigLIP pooling head
NUM_FRAMES     = 32

# ── Pre-registered sampling (do not change; see the pre-registration document) ──
SAMPLE_SEED = 20261015
VIDEOMME_CONFIRM_CHUNKS = ["02", "08", "16", "18", "20"]   # drawn at random from the 10 never-piloted chunks
MVBENCH_PER_TASK = 25                                       # 19 tasks (fine_grained_pose = NTU, excluded)
LVB_PER_GROUP = 125                                         # 4 duration groups
MLVU_PER_TASK = 72                                          # 7 multiple-choice tasks
PREREG_URL = "https://claude.ai/code/artifact/65cb336e-3916-4101-9914-8dd1ca98e189"

# ── Methods (23 compressed passes) ──
METHOD_GRID = {
    "flashvid":  [0.05, 0.10, 0.25],
    "fastvid":   [0.05, 0.10, 0.25],
    "holitom":   [0.05, 0.10, 0.25],
    "visionzip": [0.05, 0.10, 0.25],
    "scissor":   [0.05, 0.10, 0.25],
    "dycoke":    [0.9, 1.0],            # TTM pruning rate K (its lowest budgets: 35% and 28%)
    "uniform":   [0.05, 0.10, 0.25],
    "random":    [0.05, 0.10, 0.25],
}
FASTVID  = dict(c=8, tau=0.9, d=0.4, p=4, beta=0.6)
FLASHVID = dict(alpha=0.7, token_selection_method="attn_div_v2", temporal_threshold=0.8, do_segment=True,
                segment_threshold=0.9, min_segment_num=8, complementary_segment=True,
                expansion=1.25, pruning_layer=20, llm_retention_ratio=0.3)
HOLITOM  = dict(T={0.05: 0.65, 0.10: 0.65, 0.25: 0.80}, beta=0.6, D=0.0, K=7, max_window=1024,
                inner_layer=18, inner_r=0.5)
SCISSOR  = dict(epsilon=0.05, search_steps=12, refine_steps=6)

# ── Four noise passes of the full model ──
NOISE_FRAME_SHIFTS = {"frame_shift": 0.5, "frame_shift_q": 0.25}   # fraction of one sampling interval
NOISE_OPTION_ORDERS = ["option_cycle", "option_reverse"]

SEED = 0
MAX_RUNTIME_HOURS = 11.0
OUT_DIR  = f"/kaggle/working/flips_main_{BENCHMARK}"
WORK_DIR = "/tmp/astra_data"

if QUICK_TEST:
    MODEL_ID = MODEL_ID_QUICK
    NUM_FRAMES = 16
    VIDEOMME_CONFIRM_CHUNKS = ["02"]
    QUICK_MAX_Q = 6
    OUT_DIR = OUT_DIR + "_quick"
else:
    MODEL_ID = MODEL_ID_REAL
    QUICK_MAX_Q = None
assert BENCHMARK in ("videomme", "mvbench", "egoschema", "longvideobench", "mlvu"), f"Unknown BENCHMARK {BENCHMARK}"
print("Benchmark:", BENCHMARK, "| mode:", "QUICK TEST" if QUICK_TEST else "REAL RUN", "| model:", MODEL_ID,
      "| frames:", NUM_FRAMES)

### 1 · Setup

In [ ]:
# ── Setup: folders, environment, small installs ──────────────────────
import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf_home")      # keep HF cache on the big temp disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "WARN ") + "pip install " + " ".join(pkgs))

pip_install("hf_xet")   # faster Hugging Face downloads (harmless if already present)
pip_install("av")       # backup video decoder

# Same library version as the Video-MME and MVBench runs, so every benchmark runs on identical code.
# (Checked before transformers is imported, so no restart is needed.)
from importlib.metadata import version as _v, PackageNotFoundError
TRANSFORMERS_PIN = "5.16.1"
try:
    _have = _v("transformers")
except PackageNotFoundError:
    _have = None
if _have != TRANSFORMERS_PIN:
    print(f"transformers {_have} found -> installing {TRANSFORMERS_PIN} (the version used for the earlier benchmarks)")
    pip_install(f"transformers=={TRANSFORMERS_PIN}")
    assert _v("transformers") == TRANSFORMERS_PIN, "transformers pin failed — send me this log"
print("transformers", _v("transformers"), "| torch", _v("torch"), "| python", sys.version.split()[0])

try:
    from transformers import LlavaOnevisionForConditionalGeneration  # noqa: F401
    print("transformers OK")
except ImportError:
    pip_install("-U", "transformers", "accelerate")
    raise SystemExit("transformers was upgraded -> menu 'Run > Restart & Clear Cell Outputs', then 'Run All' again.")

### 1b · Run record (+ resume from an earlier session)

In [ ]:
# ── Run record + resume from an earlier session ──────────────────────
import json, datetime, glob, shutil
RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
ERR_PATH = os.path.join(OUT_DIR, "errors.log")
RUN_RECORD = {
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "benchmark": BENCHMARK, "model": MODEL_ID, "frames": NUM_FRAMES, "quick_test": QUICK_TEST,
    "versions": {"transformers": _v("transformers"), "torch": _v("torch"), "python": sys.version.split()[0]},
    "preregistration": PREREG_URL, "sample_seed": SAMPLE_SEED,
    "videomme_chunks": VIDEOMME_CONFIRM_CHUNKS, "mvbench_per_task": MVBENCH_PER_TASK,
    "lvb_per_group": LVB_PER_GROUP, "mlvu_per_task": MLVU_PER_TASK,
    "methods": METHOD_GRID, "fastvid": FASTVID, "flashvid": FLASHVID,
    "holitom": {**HOLITOM, "T": {str(k): v for k, v in HOLITOM["T"].items()}}, "scissor": SCISSOR,
    "noise": {"frame_shifts": NOISE_FRAME_SHIFTS, "option_orders": NOISE_OPTION_ORDERS},
}
with open(os.path.join(OUT_DIR, f"run_record_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"), "w") as f:
    json.dump(RUN_RECORD, f, indent=2)

# If a previous session's results.jsonl for this benchmark is attached (Add Input), continue from it.
if not os.path.exists(RES_PATH):
    # matches e.g. .../flips_main_mlvu/results.jsonl or an uploaded dataset named "flips-main-mlvu" or "mlvu-results"
    prev = [p for p in glob.glob("/kaggle/input/**/results.jsonl", recursive=True)
            if BENCHMARK in p.lower().replace("-", "_") and "quick" not in p.lower()]
    if prev:
        shutil.copy(prev[0], RES_PATH)
        print("Resuming from", prev[0])
print(json.dumps(RUN_RECORD, indent=1)[:1500])

### 2 · Imports and GPU check

In [ ]:
# ── Imports + hardware check ──────────────────────────────────────────
import json, math, time, random, shutil, zipfile, gc, heapq
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files
import transformers
from transformers import LlavaOnevisionForConditionalGeneration, AutoTokenizer

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
assert N_GPU > 0 or DATA_CHECK_ONLY, "No GPU found! Right panel > Settings > Accelerator > 'GPU T4 x2', then Run All again."
for i in range(N_GPU):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}")
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk for temp videos: {free_gb:.1f} GB")
if free_gb < 12:
    print("⚠️ Less than 12 GB free — a 5 GB video chunk needs ~11 GB while unzipping.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Optional: a Hugging Face token stored as a Kaggle Secret named HF_TOKEN (Add-ons > Secrets).
# Not needed for these public repos, but harmless.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception:
    print("No HF_TOKEN secret (fine for public repos)")

### 3 · Benchmarks: tables, pre-registered sampling, downloads, frames

In [ ]:
# ── Data: benchmark tables, pre-registered sampling, video download, frame reading ──
import io, tarfile, hashlib, re, concurrent.futures as cf
from huggingface_hub import list_repo_files

LETTERS = "ABCDEFGH"
_PREFIX = re.compile(r"^\s*\(?[A-Ha-h][\.\):]\s*")
_norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().rstrip(".").strip().lower()

def _opts(cands):
    """Plain candidate texts -> 'A. text' strings (any existing 'A.' / '(A)' prefix removed first)."""
    return [f"{LETTERS[i]}. {_PREFIX.sub('', str(c)).strip()}" for i, c in enumerate(cands)]

def _answer_letter(cands, answer):
    a = _norm(_PREFIX.sub("", str(answer)))
    hits = [i for i, c in enumerate(cands) if _norm(_PREFIX.sub("", str(c))) == a]
    return LETTERS[hits[0]] if len(hits) == 1 else None

def _hf(repo, fn, **kw):
    return hf_hub_download(repo, fn, repo_type="dataset", local_dir=os.path.join(WORK_DIR, repo.replace("/", "__")), **kw)

def _clean_cache():
    """Big archives leave a copy in the Hugging Face xet cache; remove it to keep disk free."""
    shutil.rmtree(os.path.join(os.environ.get("HF_HOME", ""), "xet"), ignore_errors=True)

def _sample(df, by, k, seed=SAMPLE_SEED):
    out = []
    for g, sub in df.groupby(by, sort=True):
        out.append(sub.sample(n=min(k, len(sub)), random_state=seed) if k else sub)
    return pd.concat(out).sort_values("qid", kind="stable").reset_index(drop=True)

# ═════ Video-MME (questions per chunk; videos come chunk by chunk) ═════
VMME_REPO = "lmms-eval/Video-MME"

def videomme_table():
    files = list_repo_files(VMME_REPO, repo_type="dataset")
    pq = [f for f in files if f.endswith(".parquet")]
    df = pd.concat([pd.read_parquet(_hf(VMME_REPO, f)) for f in pq], ignore_index=True)
    low = {c.lower(): c for c in df.columns}
    col = lambda *ns: next((low[n] for n in ns if n in low), None)
    out = pd.DataFrame({
        "video_key": df[col("videoid", "video_id")].astype(str),
        "video_alt": df[col("video_id", "videoid")].astype(str),
        "qid": df[col("question_id")].astype(str),
        "group": df[col("duration")].astype(str),
        "task": df[col("task_type")].astype(str),
        "question": df[col("question")].astype(str),
        "options": df[col("options")].apply(lambda o: [str(x) for x in list(o)]),
        "answer": df[col("answer")].astype(str).str.strip().str[0].str.upper(),
    })
    return out

VIDEO_EXT = (".mp4", ".mkv", ".webm", ".avi", ".mov")

def videomme_fetch_chunk(chunk):
    fn = f"videos_chunked_{chunk}.zip"
    t0 = time.time()
    zpath = _hf(VMME_REPO, fn)
    vdir = os.path.join(WORK_DIR, f"vmme_{chunk}")
    os.makedirs(vdir, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if m.filename.lower().endswith(VIDEO_EXT):
                with z.open(m) as src, open(os.path.join(vdir, os.path.basename(m.filename)), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
    os.remove(zpath); _clean_cache()
    vids = {os.path.splitext(f)[0]: os.path.join(vdir, f) for f in os.listdir(vdir)}
    print(f"Video-MME chunk {chunk}: {len(vids)} videos in {time.time() - t0:.0f}s")
    return vids

# ═════ MVBench (video branch: pre-cut segments, one file per video) ═════
MVB_REPO, MVB_REV = "OpenGVLab/MVBench", "video"
MVB_FOLDERS = {   # lmms-eval DATA_LIST
    "object_interaction": "star/Charades_segment", "action_sequence": "star/Charades_segment",
    "action_prediction": "star/Charades_segment", "action_localization": "sta/sta_video_segment",
    "moving_count": "clevrer/video_validation", "character_order": "perception/videos",
    "object_shuffle": "perception/videos", "egocentric_navigation": "vlnqa",
    "moving_direction": "clevrer/video_validation", "episodic_reasoning": "tvqa/video_fps3_hq_segment",
    "fine_grained_action": "Moments_in_Time_Raw/videos", "scene_transition": "scene_qa/video",
    "state_change": "perception/videos", "moving_attribute": "clevrer/video_validation",
    "action_antonym": "ssv2_video_mp4", "unexpected_action": "FunQA_test/test",
    "counterfactual_inference": "clevrer/video_validation", "object_existence": "clevrer/video_validation",
    "action_count": "perception/videos",
}   # fine_grained_pose (NTU RGB+D) is excluded by the pre-registration

def _read_json_any(path):
    txt = open(path, encoding="utf-8").read().strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(l) for l in txt.splitlines() if l.strip()]

MVB_FILES = None
def mvbench_table():
    global MVB_FILES
    MVB_FILES = list_repo_files(MVB_REPO, repo_type="dataset", revision=MVB_REV)
    rows, dropped = [], 0
    for task, folder in MVB_FOLDERS.items():
        jf = next((f for f in MVB_FILES if f.endswith(f"json/{task}.json")), f"json/{task}.json")
        for i, d in enumerate(_read_json_any(_hf(MVB_REPO, jf, revision=MVB_REV))):
            ans = _answer_letter(d["candidates"], d["answer"])
            if ans is None or len(d["candidates"]) > len(LETTERS):
                dropped += 1
                continue
            rows.append({"qid": f"{task}-{i:04d}", "video_key": f"{folder}/{d['video']}", "video_alt": "",
                         "group": task, "task": task, "question": str(d["question"]),
                         "options": _opts(d["candidates"]), "answer": ans})
    print(f"MVBench: {len(rows)} questions in {len(MVB_FOLDERS)} tasks ({dropped} dropped: answer not among candidates)")
    return pd.DataFrame(rows)

def mvbench_fetch(keys):
    all_files, by_base = set(MVB_FILES), {}
    for f in MVB_FILES:
        by_base.setdefault(os.path.basename(f), []).append(f)
    def one(k):
        cand = [k] if k in all_files else [f for f in by_base.get(os.path.basename(k), [])
                                         if f.startswith(k.split("/")[0])] or by_base.get(os.path.basename(k), [])
        if not cand:
            raise FileNotFoundError(k)
        return k, _hf(MVB_REPO, cand[0], revision=MVB_REV)
    return _parallel_fetch(keys, one)

# ═════ EgoSchema (public 500-question subset) ═════
EGO_REPO = "lmms-lab/egoschema"
def egoschema_table():
    files = list_repo_files(EGO_REPO, repo_type="dataset")
    pq = [f for f in files if f.startswith("Subset/") and f.endswith(".parquet")]
    df = pd.concat([pd.read_parquet(_hf(EGO_REPO, f)) for f in pq], ignore_index=True)
    rows = []
    for _, d in df.iterrows():
        opts = [str(o) for o in list(d["option"])]
        rows.append({"qid": str(d["question_idx"]), "video_key": str(d["video_idx"]), "video_alt": "",
                     "group": "ego", "task": "ego", "question": str(d["question"]), "options": _opts(opts),
                     "answer": LETTERS[int(str(d["answer"]).strip())]})
    print(f"EgoSchema subset: {len(rows)} questions")
    return pd.DataFrame(rows)

def egoschema_fetch(keys):
    need, got = set(keys), {}
    files = sorted(f for f in list_repo_files(EGO_REPO, repo_type="dataset") if f.endswith(".zip"))
    vdir = os.path.join(WORK_DIR, "ego_videos"); os.makedirs(vdir, exist_ok=True)
    for fn in files:
        if not need - set(got):
            break
        t0 = time.time()
        zpath = _hf(EGO_REPO, fn)
        with zipfile.ZipFile(zpath) as z:
            for m in z.infolist():
                stem = os.path.splitext(os.path.basename(m.filename))[0]
                if stem in need and stem not in got and m.filename.lower().endswith(VIDEO_EXT):
                    dst = os.path.join(vdir, os.path.basename(m.filename))
                    with z.open(m) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out, 1 << 24)
                    got[stem] = dst
        os.remove(zpath); _clean_cache()
        print(f"  {fn}: {len(got)}/{len(need)} videos found ({time.time() - t0:.0f}s, "
              f"free disk {shutil.disk_usage(WORK_DIR).free / 1e9:.0f} GB)")
    return got

# ═════ LongVideoBench (validation, video only) ═════
LVB_REPO = "longvideobench/LongVideoBench"
def longvideobench_table():
    files = list_repo_files(LVB_REPO, repo_type="dataset")
    if "lvb_val.json" in files:
        data = _read_json_any(_hf(LVB_REPO, "lvb_val.json"))
    else:
        data = pd.read_parquet(_hf(LVB_REPO, next(f for f in files if f.startswith("validation")))).to_dict("records")
    rows = []
    for d in data:
        cands = d.get("candidates")
        if cands is None:
            cands = [d[f"option{i}"] for i in range(5) if d.get(f"option{i}") not in (None, "N/A")]
        cands = [str(c) for c in list(cands)]
        rows.append({"qid": str(d["id"]), "video_key": os.path.basename(str(d["video_path"])), "video_alt": "",
                     "group": str(d["duration_group"]), "task": str(d.get("question_category", "")),
                     "question": str(d["question"]), "options": _opts(cands),
                     "answer": LETTERS[int(d["correct_choice"])]})
    print(f"LongVideoBench val: {len(rows)} questions")
    return pd.DataFrame(rows)

class _PartsReader(io.RawIOBase):
    """Reads videos.tar.part.aa, .ab, ... as one stream; downloads each part when needed, deletes it after."""
    def __init__(self, parts):
        self.parts, self.k, self.f, self.path = list(parts), -1, None, None
    def readable(self):
        return True
    def _next(self):
        if self.f:
            self.f.close(); os.remove(self.path); _clean_cache()
        self.k += 1
        if self.k >= len(self.parts):
            self.f = None
            return False
        t0 = time.time()
        self.path = _hf(LVB_REPO, self.parts[self.k])
        self.f = open(self.path, "rb")
        print(f"  {self.parts[self.k]} ready ({time.time() - t0:.0f}s)")
        return True
    def readinto(self, b):
        while True:
            if self.f is None and not self._next():
                return 0
            n = self.f.readinto(b)
            if n:
                return n
            if not self._next():
                return 0

def longvideobench_fetch(keys):
    need, got = set(keys), {}
    parts = sorted(f for f in list_repo_files(LVB_REPO, repo_type="dataset") if f.startswith("videos.tar.part"))
    vdir = os.path.join(WORK_DIR, "lvb_videos"); os.makedirs(vdir, exist_ok=True)
    reader = io.BufferedReader(_PartsReader(parts), buffer_size=1 << 24)
    with tarfile.open(fileobj=reader, mode="r|") as tar:
        for m in tar:
            b = os.path.basename(m.name)
            if m.isfile() and b in need and b not in got:
                with tar.extractfile(m) as src, open(os.path.join(vdir, b), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
                got[b] = os.path.join(vdir, b)
            if not need - set(got):
                break
    return got

# ═════ MLVU (dev, the 7 multiple-choice tasks; one file per video) ═════
MLVU_REPO = "MLVU/MVLU"
MLVU_TASKS = ["1_plotQA", "2_needle", "3_ego", "4_count", "5_order", "6_anomaly_reco", "7_topic_reasoning"]
MLVU_FILES = None
def mlvu_table():
    global MLVU_FILES
    MLVU_FILES = list_repo_files(MLVU_REPO, repo_type="dataset")
    rows, dropped = [], 0
    for task in MLVU_TASKS:
        jf = next(f for f in MLVU_FILES if f.endswith(f"json/{task}.json"))
        for i, d in enumerate(_read_json_any(_hf(MLVU_REPO, jf))):
            ans = _answer_letter(d["candidates"], d["answer"])
            if ans is None or len(d["candidates"]) > len(LETTERS):
                dropped += 1
                continue
            rows.append({"qid": f"{task}-{i:04d}", "video_key": f"{task}/{d['video']}", "video_alt": "",
                         "group": task, "task": task, "question": str(d["question"]),
                         "options": _opts(d["candidates"]), "answer": ans})
    print(f"MLVU dev MC: {len(rows)} questions ({dropped} dropped)")
    return pd.DataFrame(rows)

def mlvu_fetch(keys):
    by_rel = {"/".join(f.split("/")[-2:]): f for f in MLVU_FILES if f.lower().endswith(VIDEO_EXT)}
    def one(k):
        if k not in by_rel:
            raise FileNotFoundError(k)
        return k, _hf(MLVU_REPO, by_rel[k])
    return _parallel_fetch(keys, one)

def _parallel_fetch(keys, one, workers=8):
    got, t0 = {}, time.time()
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(one, k): k for k in sorted(set(keys))}
        for i, fu in enumerate(cf.as_completed(futs)):
            try:
                k, p = fu.result()
                got[k] = p
            except Exception as e:
                with open(ERR_PATH, "a") as f:
                    f.write(f"{futs[fu]}\tdownload\t{type(e).__name__}: {e}\n")
            if (i + 1) % 100 == 0:
                print(f"  downloaded {i + 1}/{len(futs)} ({time.time() - t0:.0f}s)")
    print(f"Downloaded {len(got)}/{len(futs)} videos in {time.time() - t0:.0f}s")
    return got

# ═════ Manifest: the pre-registered sample, frozen to a file with its hash ═════
def _valid(df):
    """Pre-registration: questions with fewer than two options carry no information and are removed before sampling."""
    bad = df.options.apply(len) < 2
    if bad.any():
        print(f"Removed {int(bad.sum())} question(s) with fewer than two options: {df.qid[bad].tolist()}")
    return df[~bad]

def build_manifest():
    if BENCHMARK == "videomme":
        df = _valid(videomme_table())              # filtered to the confirmatory chunks as they are downloaded
    elif BENCHMARK == "mvbench":
        df = _sample(_valid(mvbench_table()), "task", MVBENCH_PER_TASK)
    elif BENCHMARK == "egoschema":
        df = _valid(egoschema_table())
    elif BENCHMARK == "longvideobench":
        df = _sample(_valid(longvideobench_table()), "group", LVB_PER_GROUP)
    else:
        df = _sample(_valid(mlvu_table()), "task", MLVU_PER_TASK)
    if BENCHMARK != "videomme":
        ids = df.qid.tolist()
        h = hashlib.sha256("\n".join(ids).encode()).hexdigest()
        with open(os.path.join(OUT_DIR, f"manifest_{BENCHMARK}.json"), "w") as f:
            json.dump({"benchmark": BENCHMARK, "seed": SAMPLE_SEED, "n": len(ids), "sha256": h, "qids": ids}, f)
        print(f"Manifest: {len(ids)} questions, {df.video_key.nunique()} videos, sha256 {h[:16]}…")
        print("Per group:", df.group.value_counts().sort_index().to_dict())
    return df

# ═════ Frames ═════
def _frame_indices(total, n, shift):
    """Uniform sampling as in LMMs-Eval (np.linspace over all frames). shift = fraction of one sampling
    interval to move every sample later (noise passes)."""
    idx = np.linspace(0, total - 1, n)
    if shift:
        idx = idx + shift * (total - 1) / max(n - 1, 1)
    return np.clip(idx, 0, total - 1).round().astype(int)

def _read_cv2(path, n, shift=0.0):
    import cv2
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total > 0:
        for k in _frame_indices(total, n, shift):
            ok = False
            for off in (0, -1, 1, -5, 5, -20):
                kk = int(np.clip(k + off, 0, total - 1))
                cap.set(cv2.CAP_PROP_POS_FRAMES, kk)
                ok, fr = cap.read()
                if ok:
                    break
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def _read_av(path, n, shift=0.0):
    import av
    out = []
    with av.open(path) as c:
        s = c.streams.video[0]
        dur = float(s.duration * s.time_base) if s.duration else float(c.duration or 0) / 1e6
        end = max(dur - 0.5, 0)
        ts = np.linspace(0, end, n)
        if shift:
            ts = np.clip(ts + shift * end / max(n - 1, 1), 0, end)
        for t in ts:
            c.seek(int(t / s.time_base), stream=s, any_frame=False, backward=True)
            for fr in c.decode(s):
                out.append(fr.to_ndarray(format="rgb24"))
                break
    return out

def read_frames(path, n, shift=0.0):
    frames = []
    try:
        frames = _read_cv2(path, n, shift)
    except Exception as e:
        print("cv2 failed:", e)
    if len(frames) < n:
        try:
            frames = _read_av(path, n, shift)
        except Exception as e:
            print("av failed:", e)
    if len(frames) == 0:
        raise RuntimeError(f"Could not decode {path}")
    while len(frames) < n:
        frames.append(frames[-1])
    return frames[:n]

def read_all_frame_sets(path):
    """The three frame sets every video needs: original, and the two shifted noise passes."""
    return {"orig": read_frames(path, NUM_FRAMES),
            **{k: read_frames(path, NUM_FRAMES, shift=s) for k, s in NOISE_FRAME_SHIFTS.items()}}

# ═════ Option-order noise passes ═════
def _new_index(kind, j, n):
    """Which ORIGINAL option is shown at new position j."""
    return (j + 1) % n if kind == "option_cycle" else n - 1 - j

def reorder_options(options, kind):
    body = [_PREFIX.sub("", str(o)).strip() for o in options]
    n = len(body)
    return [f"{LETTERS[j]}. {body[_new_index(kind, j, n)]}" for j in range(n)]

def map_back(res, n, kind):
    """Express a reordered result in the original option letters."""
    lp_new = np.array(res["logp"])
    lp = np.empty(n)
    for j in range(n):
        lp[_new_index(kind, j, n)] = lp_new[j]
    srt = np.sort(lp)[::-1]
    out = dict(res)
    out.update(pred=LETTERS[int(lp.argmax())], logp=lp.round(4).tolist(), margin=float(srt[0] - srt[1]))
    return out


# ═════ Optional: CPU-only data check of all five benchmarks (DATA_CHECK_ONLY = True) ═════
def data_check():
    global BENCHMARK
    report, keep = {}, BENCHMARK
    for b in ["videomme", "mvbench", "egoschema", "longvideobench", "mlvu"]:
        BENCHMARK = b
        t0, r = time.time(), {}
        try:
            full = {"videomme": videomme_table, "mvbench": mvbench_table, "egoschema": egoschema_table,
                    "longvideobench": longvideobench_table, "mlvu": mlvu_table}[b]()
            r["questions_total"] = len(full)
            df = build_manifest()
            r["questions_sampled"], r["videos"] = len(df), int(df.video_key.nunique())
            r["options_per_question"] = df.options.apply(len).value_counts().sort_index().to_dict()
            r["answer_letters"] = df.answer.value_counts().sort_index().to_dict()
            keys = df.video_key.drop_duplicates().head(2).tolist()
            if b == "videomme":
                vids = videomme_fetch_chunk(VIDEOMME_CONFIRM_CHUNKS[0])
                sub = df[df.video_key.isin(vids) | df.video_alt.isin(vids)]
                r["chunk_questions"] = len(sub)
                paths = {k: vids.get(k) for k in sub.video_key.head(2)}
            elif b in ("mvbench", "mlvu"):
                paths = (mvbench_fetch if b == "mvbench" else mlvu_fetch)(keys)
            elif b == "egoschema":
                zf = sorted(f for f in list_repo_files(EGO_REPO, repo_type="dataset") if f.endswith(".zip"))[0]
                zpath = _hf(EGO_REPO, zf)
                with zipfile.ZipFile(zpath) as z:
                    stems = {os.path.splitext(os.path.basename(m.filename))[0]: m for m in z.infolist()
                             if m.filename.lower().endswith(VIDEO_EXT)}
                    hit = [k for k in df.video_key if k in stems][:2]
                    r["first_zip_videos"], r["first_zip_needed"] = len(stems), int(df.video_key.isin(stems).sum())
                    paths = {}
                    for k in hit:
                        dst = os.path.join(WORK_DIR, os.path.basename(stems[k].filename))
                        with z.open(stems[k]) as src, open(dst, "wb") as out:
                            shutil.copyfileobj(src, out, 1 << 24)
                        paths[k] = dst
                os.remove(zpath); _clean_cache()
            else:
                part = sorted(f for f in list_repo_files(LVB_REPO, repo_type="dataset") if f.startswith("videos.tar.part"))[0]
                ppath = _hf(LVB_REPO, part)
                names, paths, need = [], {}, set(df.video_key)
                with open(ppath, "rb") as fh, tarfile.open(fileobj=fh, mode="r|") as tar:
                    try:
                        for m in tar:
                            names.append(m.name)
                            b_ = os.path.basename(m.name)
                            if m.isfile() and b_ in need and len(paths) < 2:
                                dst = os.path.join(WORK_DIR, b_)
                                with tar.extractfile(m) as src, open(dst, "wb") as out:
                                    shutil.copyfileobj(src, out, 1 << 24)
                                paths[b_] = dst
                    except (tarfile.TarError, EOFError, OSError):
                        pass                              # the part ends mid-file; expected
                r["first_part_members"] = len(names)
                r["first_part_example_names"] = names[:3]
                os.remove(ppath); _clean_cache()
            r["decoded"] = {}
            for k, p in paths.items():
                fr = read_all_frame_sets(p)
                r["decoded"][k] = {kk: [len(v), list(v[0].shape)] for kk, v in fr.items()}
            r["status"] = "OK" if r["decoded"] else "NO VIDEO DECODED"
        except Exception as e:
            r["status"] = f"FAIL {type(e).__name__}: {e}"
        r["seconds"] = round(time.time() - t0)
        report[b] = r
        print(f"\n### {b}: {r['status']}\n{json.dumps(r, indent=1, default=str)[:1500]}")
    BENCHMARK = keep
    with open(os.path.join(OUT_DIR, "data_check.json"), "w") as f:
        json.dump(report, f, indent=2, default=str)
    return report

if DATA_CHECK_ONLY:
    data_check()
    raise SystemExit("Data check finished — send me data_check.json (this stop is expected).")

### 4 · Model, T4 memory fix, our decoder loop, prefix caching

In [ ]:
# ── Model: LLaVA-OneVision with direct control over visual tokens ──────
# ── T4 memory fix + safe attention routing ─────────────────────────────
# 1) Expand grouped K/V heads ourselves so PyTorch can use the memory-efficient kernel (math kernel -> OOM on T4).
# 2) Never hand an explicit mask to the memory-efficient kernel: on T4 that path can crash with
#    "illegal memory access". A plain causal mask is converted to is_causal=True (identical maths).
# 3) Guard: if any NON-causal mask (e.g. transformers mistaking position gaps for packed sequences)
#    reaches a long attention call, stop loudly instead of computing something silently wrong.
from torch.nn.attention import sdpa_kernel, SDPBackend
if not hasattr(torch.nn.functional, "_astra_orig_sdpa"):
    torch.nn.functional._astra_orig_sdpa = torch.nn.functional.scaled_dot_product_attention
_orig_sdpa = torch.nn.functional._astra_orig_sdpa
_TRIL_CACHE = {}
def _plain_causal(mask, L, S):
    if L != S or mask.dim() != 4 or mask.shape[0] != 1 or mask.shape[1] != 1:
        return False
    key = (L, str(mask.device))
    if key not in _TRIL_CACHE:
        _TRIL_CACHE.clear()
        _TRIL_CACHE[key] = torch.ones(L, L, dtype=torch.bool, device=mask.device).tril()
    allowed = mask[0, 0] if mask.dtype == torch.bool else (mask[0, 0] == 0)
    return torch.equal(allowed, _TRIL_CACHE[key])

def _sdpa_safe(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False, **kw):
    if enable_gqa and key.shape[-3] != query.shape[-3]:
        rep = query.shape[-3] // key.shape[-3]
        key = key.repeat_interleave(rep, dim=-3)
        value = value.repeat_interleave(rep, dim=-3)
    if attn_mask is not None:
        L, S = query.shape[-2], key.shape[-2]
        if _plain_causal(attn_mask, L, S):
            attn_mask, is_causal = None, True
        elif L > 1024:
            raise RuntimeError("A non-causal attention mask reached a long attention call — stopping instead of "
                               "computing silently wrong results. Send me this error.")
        elif query.dtype == torch.float16:
            # short question block over a cached prefix: compute in fp32 (cheap at this size), so cached passes
            # match the fp32-accumulating kernel used for full passes as closely as possible
            m = attn_mask if attn_mask.dtype == torch.bool else attn_mask.float()
            return _orig_sdpa(query.float(), key.float(), value.float(), attn_mask=m, dropout_p=dropout_p,
                              is_causal=is_causal, scale=scale, **kw).to(query.dtype)
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale, **kw)

torch.nn.functional.scaled_dot_product_attention = _sdpa_safe
NO_MATH = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION]
print("T4 attention fix applied (K/V heads expanded, causal masks routed to is_causal, math kernel disabled for the LLM)")

max_memory = {i: "13GiB" for i in range(N_GPU)}
if N_GPU >= 2:
    max_memory[0] = "10GiB"
_dtype_kw = {"dtype": torch.float16} if int(transformers.__version__.split(".")[0]) >= 5 else {"torch_dtype": torch.float16}
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", max_memory=max_memory, attn_implementation="sdpa", **_dtype_kw).eval()
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def _get(root, paths):
    for p in paths:
        obj, ok = root, True
        for part in p.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError(f"None of {paths} found — send me the error and transformers version")

vision_tower = _get(model, ["vision_tower", "model.vision_tower"])
projector    = _get(model, ["multi_modal_projector", "model.multi_modal_projector"])
NEWLINE      = _get(model, ["image_newline", "model.image_newline"]).data
lm_head      = _get(model, ["lm_head", "language_model.lm_head"])
body = None
for _, mod in model.named_modules():
    if hasattr(mod, "embed_tokens") and hasattr(mod, "layers") and hasattr(mod, "norm"):
        body = mod
        break
assert body is not None, "Could not find the language decoder"

CFG, VCFG, TCFG = model.config, model.config.vision_config, model.config.text_config
VFL = CFG.vision_feature_layer
VSS = getattr(CFG, "vision_feature_select_strategy", "full")
IMG_SIZE, PATCH = VCFG.image_size, VCFG.patch_size
SIDE = IMG_SIZE // PATCH
POOLED = math.ceil(SIDE / 2)
NLAYERS, HID = TCFG.num_hidden_layers, TCFG.hidden_size
NHEADS, NKV = TCFG.num_attention_heads, TCFG.num_key_value_heads
HEAD_DIM = getattr(TCFG, "head_dim", None) or HID // NHEADS
INTER = TCFG.intermediate_size
EMB_DEVICE = body.embed_tokens.weight.device
LM_DEVICE = lm_head.weight.device
try:
    from transformers import AutoImageProcessor
    _ip = AutoImageProcessor.from_pretrained(MODEL_ID)
    IMG_MEAN, IMG_STD = list(_ip.image_mean), list(_ip.image_std)
except Exception:
    IMG_MEAN, IMG_STD = [0.5] * 3, [0.5] * 3

# ── FastVID needs the ORIGINAL SigLIP post-layernorm + attention-pooling head (as in the official code) ──
from transformers import SiglipVisionModel
_sig = SiglipVisionModel.from_pretrained(SIGLIP_ID, **_dtype_kw)
VDEV = next(vision_tower.parameters()).device
# transformers v4 nests these under .vision_model; v5 puts them directly on the model
SIG_POSTLN = _get(_sig, ["vision_model.post_layernorm", "post_layernorm", "model.post_layernorm"]).to(VDEV).eval()
SIG_HEAD = _get(_sig, ["vision_model.head", "head", "model.head"]).to(VDEV).eval()
for _need in ("probe", "attention", "layernorm", "mlp"):
    assert hasattr(SIG_HEAD, _need), (f"SigLIP head has no '{_need}' (type {type(SIG_HEAD).__name__}, "
                                      f"children {[n for n, _ in SIG_HEAD.named_children()]}) — send me this error")
assert isinstance(SIG_HEAD.attention, torch.nn.MultiheadAttention), \
    f"Unexpected SigLIP head attention type {type(SIG_HEAD.attention).__name__} — send me this error"
print("SigLIP pooling head loaded (post_layernorm + attention-pooling head) ✅")

# ── VisionZip needs the attention and keys of the vision layer that produces the features ──
# (official LLaVA-OV port in the HoliTom repo: attentions[-1].mean(heads).mean(queries) and keys.mean(heads))
ENC_LAYERS = _get(vision_tower, ["vision_model.encoder.layers", "encoder.layers", "model.encoder.layers"])
assert isinstance(VFL, int), "VisionZip port assumes a single vision feature layer"
_n_hs = len(ENC_LAYERS) + 1
_h_idx = VFL if VFL >= 0 else _n_hs + VFL
VZ_LAYER, VZ_IN = ENC_LAYERS[_h_idx - 1], _h_idx - 1
VZ_HEADS = VCFG.num_attention_heads
VZ_HD = VCFG.hidden_size // VZ_HEADS
print(f"VisionZip reads attention/keys from vision layer {_h_idx - 1} of {len(ENC_LAYERS)}")
del _sig; gc.collect()
print(f"LLM layers={NLAYERS} hidden={HID} heads={NHEADS}/{NKV} | {SIDE}x{SIDE} -> {POOLED}x{POOLED} tokens/frame")

LETTERS = "ABCDEFGH"
LETTER_IDS = []
for L in LETTERS:
    ids = tok.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"Letter {L} is not a single token"
    LETTER_IDS.append(ids[0])

PREFIX = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n"
PREFIX_IDS = tok(PREFIX, add_special_tokens=False, return_tensors="pt").input_ids[0]

def _letter_list(n):
    L = list(LETTERS[:n])
    return L[0] if n == 1 else (f"{L[0]} or {L[1]}" if n == 2 else ", ".join(L[:-1]) + ", or " + L[-1])

def build_suffix_ids(question, options):
    """The pilot prompt; for 4 options the text is identical to Steps 1 and 1b."""
    opts = "\n".join(str(o) for o in options)
    s = ("\nSelect the best answer to the following multiple-choice question based on the video. "
         f"Respond with only the letter ({_letter_list(len(options))}) of the correct option.\n"
         f"{question}\n{opts}\nThe best answer is:<|im_end|>\n<|im_start|>assistant\n")
    return tok(s, add_special_tokens=False, return_tensors="pt").input_ids[0]

def preprocess(frames):
    arr = np.stack([np.asarray(Image.fromarray(f).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))
                    for f in frames]).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(0, 3, 1, 2)
    m, s = torch.tensor(IMG_MEAN)[:, None, None], torch.tensor(IMG_STD)[:, None, None]
    return (x - m) / s

@torch.no_grad()
def encode_video(pix, bs=8):
    """frames -> (T, POOLED*POOLED, D) projected tokens (exactly LLaVA-OV's video path, before the newline token),
    FastVID's per-frame SigLIP-head attention (T, P') and global feature (T, 1152),
    and VisionZip's pooled vision attention (T, P') and pooled keys (T, P', head_dim)."""
    pdev = next(projector.parameters()).device
    feats, attns, globs, vz_a, vz_m = [], [], [], [], []
    for i in range(0, pix.shape[0], bs):
        out = vision_tower(pixel_values=pix[i:i+bs].to(VDEV, torch.float16), output_hidden_states=True)
        h = out.hidden_states[VFL]
        if VSS == "default":
            h = h[:, 1:]
        # FastVID "vision abstract": original SigLIP post_layernorm + attention-pooling head on the same features
        z = SIG_POSTLN(h.to(VDEV))
        probe = SIG_HEAD.probe.repeat(z.shape[0], 1, 1)
        att_out, att_w = SIG_HEAD.attention(probe, z, z, need_weights=True)      # att_w: (b, 1, SIDE*SIDE)
        pooled = att_out + SIG_HEAD.mlp(SIG_HEAD.layernorm(att_out))
        globs.append(pooled[:, 0].float())
        attns.append(att_w.float())
        # VisionZip: self-attention of the feature-producing layer, recomputed (sdpa does not return weights)
        xin = out.hidden_states[VZ_IN].to(VZ_LAYER.layer_norm1.weight.device)
        if VSS == "default":
            xin = xin[:, 1:]
        xin = VZ_LAYER.layer_norm1(xin)
        b, P, _ = xin.shape
        q = VZ_LAYER.self_attn.q_proj(xin).float().view(b, P, VZ_HEADS, VZ_HD).transpose(1, 2)
        k = VZ_LAYER.self_attn.k_proj(xin).float().view(b, P, VZ_HEADS, VZ_HD).transpose(1, 2)
        w = torch.softmax(torch.matmul(q, k.transpose(2, 3)) * VZ_HD ** -0.5, dim=-1)
        vz_a.append(w.mean(dim=1).mean(dim=1))          # (b, P): attention each token receives
        vz_m.append(k.mean(dim=1))                      # (b, P, head_dim): keys averaged over heads
        del q, k, w
        feats.append(projector(h.to(pdev)))
    def pool(t):                                        # (T, SIDE*SIDE, C) -> (T, POOLED*POOLED, C), = get_2dPool
        T, P, C = t.shape
        t = t.view(T, SIDE, SIDE, C).permute(0, 3, 1, 2).float()
        t = F.interpolate(t, size=(POOLED, POOLED), mode="bilinear")
        return t.permute(0, 2, 3, 1).reshape(T, POOLED * POOLED, C)
    f = pool(torch.cat(feats)).contiguous().to(torch.float16)
    dev = f.device
    a = pool(torch.cat(attns).view(-1, SIDE * SIDE, 1).to(dev))[..., 0]
    va = pool(torch.cat(vz_a).unsqueeze(-1).to(dev))[..., 0]
    vm = pool(torch.cat(vz_m).to(dev))
    return f, a, torch.cat(globs).to(dev), va, vm

def llm_flops_layers(ns):
    """Prefill FLOPs of the LLM when layer l sees ns[l] tokens."""
    dkv = NKV * HEAD_DIM
    return float(sum(2 * (2 * n * HID * HID + 2 * n * HID * dkv + 2 * n * n * HID + 3 * n * HID * INTER) for n in ns))

def llm_flops(n):
    return llm_flops_layers([n] * NLAYERS)

def _sync():
    for i in range(N_GPU):
        torch.cuda.synchronize(i)

# ── inner-LLM operations (FlashVID: FastV-style pruning; HoliTom: similarity merging) ──
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb, repeat_kv

@torch.no_grad()
def last_query_attention(layer, h, pe):
    """Attention of the LAST token over all tokens in this layer, (1, heads, 1, N) — exactly the row both official
    codes read (FlashVID computes it this way under flash/sdpa; HoliTom takes the last row of eager attention)."""
    a = layer.self_attn
    dev = a.q_proj.weight.device
    h = h.to(dev)
    cos, sin = pe[0].to(dev), pe[1].to(dev)
    hn = layer.input_layernorm(h)
    q = a.q_proj(hn[:, -1:]).view(1, 1, -1, HEAD_DIM).transpose(1, 2)
    k = a.k_proj(hn).view(1, hn.shape[1], -1, HEAD_DIM).transpose(1, 2)
    q, _ = apply_rotary_pos_emb(q, q, cos[:, -1:], sin[:, -1:])
    _, k = apply_rotary_pos_emb(k, k, cos, sin)
    k = repeat_kv(k, NHEADS // NKV)
    w = torch.matmul(q, k.transpose(2, 3)) * a.scaling
    return torch.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)

@torch.no_grad()
def apply_inner(h, pos, pe, attn, inner, v0, vlen, has_newline):
    """Returns pruned/merged (h, pos, pe) and the number of visual tokens left."""
    dev = h.device
    a = attn.mean(dim=1)[0, -1].to(dev)                       # mean over heads -> (N,)
    N = h.shape[1]
    if inner["kind"] == "fastv":                               # FlashVID utils.fastv_prune
        n_keep = math.ceil(vlen * inner["keep_ratio"])
        top = torch.topk(a[v0:v0 + vlen], k=n_keep).indices.sort().values + v0
        keep = torch.cat([torch.arange(v0, device=dev), top, torch.arange(v0 + vlen, N, device=dev)]).sort().values
        n_vis = n_keep
    elif inner["kind"] == "holitom":                           # HoliTom modeling_qwen2 (newline counted in the span)
        L = vlen + (1 if (has_newline and inner["include_newline"]) else 0)
        n_top = round(L * (1 - inner["merge_ratio"]))
        top = a[v0:v0 + L].topk(n_top).indices + v0
        span = torch.arange(v0, v0 + L, device=dev)
        non_top = span[~torch.isin(span, top)]                 # ascending, as in the official loop
        hs = h[0].float()
        nt, tp = hs[non_top], hs[top]
        sim = (nt @ tp.t()) / (nt.norm(dim=-1, keepdim=True) * tp.norm(dim=-1, keepdim=True).t())
        tgt = sim.argmax(dim=-1)
        # official: for each non-top token in order, h[t] = (h[t] + h[i]) / 2  -> closed form below (same maths)
        m = torch.bincount(tgt, minlength=len(top))
        srt, perm = torch.sort(tgt, stable=True)
        first = torch.searchsorted(srt, srt)
        rank = torch.empty_like(tgt)
        rank[perm] = torch.arange(len(tgt), device=dev) - first + 1
        w = torch.pow(2.0, -(m[tgt] - rank + 1).float())
        new_top = tp * torch.pow(2.0, -m.float()).unsqueeze(1)
        new_top.index_add_(0, tgt, nt * w.unsqueeze(1))
        h = h.clone()
        h[0, top] = new_top.to(h.dtype)
        keep = torch.cat([torch.arange(v0, device=dev), top, torch.arange(v0 + L, N, device=dev)]).sort().values
        n_vis = n_top - (int(bool((top == v0 + vlen).any())) if L > vlen else 0)
    else:
        raise ValueError(inner["kind"])
    kp = keep.to(pos.device)
    pe = (pe[0][:, keep.to(pe[0].device)], pe[1][:, keep.to(pe[1].device)])
    return h[:, keep], pos[:, kp], pe, n_vis

@torch.no_grad()
def forward_layers(x, pos, inner=None, v0=0, vlen=0, has_newline=True):
    """Our own decoder loop (same calls as Qwen2Model.forward); lets FlashVID/HoliTom act inside the LLM."""
    pe = body.rotary_emb(x, pos)
    h, attn, ns, nvis = x, None, [], []
    vis_now = vlen
    for li, layer in enumerate(body.layers):
        if inner is not None and li == inner["layer"]:
            h, pos, pe, vis_now = apply_inner(h, pos, pe, attn, inner, v0, vlen, has_newline)
            attn = None
        if inner is not None and li == inner["layer"] - 1:
            attn = last_query_attention(layer, h, pe)
        ns.append(h.shape[1]); nvis.append(vis_now)
        h = layer(h, attention_mask=None, position_ids=pos, position_embeddings=pe, use_cache=False)
        if isinstance(h, tuple):
            h = h[0]
    h = body.norm(h.to(body.norm.weight.device))
    return h, ns, nvis

@torch.no_grad()
def llm_pass(vis, vis_pos, n_video_full, suf_ids, n_opt, pre_ids=None, newline=True, inner=None, manual=False):
    """One prefill. vis: kept video tokens (K, D); vis_pos: their position offsets inside the video span;
    n_video_full: length of the video span for positions (the newline and question follow it).
    newline=False drops LLaVA-OV's video newline token (LLaVA-Scissor's flat path).
    inner: optional FlashVID/HoliTom inner-LLM operation (runs our own decoder loop)."""
    E = body.embed_tokens
    pre_ids = PREFIX_IDS if pre_ids is None else pre_ids
    pre, suf = E(pre_ids.to(EMB_DEVICE)), E(suf_ids.to(EMB_DEVICE))
    parts = [pre, vis.to(EMB_DEVICE, pre.dtype)]
    if newline:
        parts.append(NEWLINE.to(EMB_DEVICE, pre.dtype)[None])
    parts.append(suf)
    x = torch.cat(parts, 0)[None]
    n_pre, n_q = pre.shape[0], suf.shape[0]
    vp = torch.as_tensor(vis_pos, device=EMB_DEVICE, dtype=torch.long)
    nl = 1 if newline else 0
    pos = torch.cat([torch.arange(n_pre, device=EMB_DEVICE), n_pre + vp] +
                    ([torch.tensor([n_pre + n_video_full], device=EMB_DEVICE)] if newline else []) +
                    [n_pre + n_video_full + nl + torch.arange(n_q, device=EMB_DEVICE)])[None]
    am = torch.ones(1, x.shape[1], dtype=torch.long, device=EMB_DEVICE)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    _sync(); t0 = time.time()
    with sdpa_kernel(NO_MATH):
        if inner is None and not manual:
            out = body(inputs_embeds=x, attention_mask=am, position_ids=pos, use_cache=False)
            h = out[0][0, -1]
            ns, nvis = [x.shape[1]] * NLAYERS, [vis.shape[0]] * NLAYERS
        else:
            hh, ns, nvis = forward_layers(x, pos, inner, v0=n_pre, vlen=vis.shape[0], has_newline=newline)
            h = hh[0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    lp = torch.log_softmax(logits[LETTER_IDS[:n_opt]], -1).cpu().numpy()
    if not np.all(np.isfinite(lp)):
        raise FloatingPointError("NaN/inf logits (fp16 overflow)")
    srt = np.sort(lp)[::-1]
    return {"pred": LETTERS[int(lp.argmax())], "logp": lp.round(4).tolist(), "margin": float(srt[0] - srt[1]),
            "ntok": int(vis.shape[0]), "vis_mean_layers": float(np.mean(nvis)), "sec": float(dt),
            "flops": llm_flops_layers(ns),
            "peak_gb": round(max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9, 3)}

# ── Prefix caching: the system prompt + video tokens are run once per (video, pass); every question reuses them ──
from transformers import DynamicCache

def _new_cache():
    try:
        return DynamicCache(config=TCFG)
    except TypeError:
        return DynamicCache()

def _crop(cache, n):
    extra = cache.get_seq_length() - n
    if extra > 0 and hasattr(cache, "crop"):
        cache.crop(-extra)                      # negative = drop that many tokens (works in old and new versions)
    if cache.get_seq_length() != n:
        for l in cache.layers:
            l.keys, l.values = l.keys[..., :n, :], l.values[..., :n, :]
    assert cache.get_seq_length() == n, "could not restore the cached prefix"

@torch.no_grad()
def prefill(vis, vis_pos, n_video_full, newline=True, pre_ids=None):
    """Runs [system prompt, video tokens, (newline)] once and keeps the key/value cache."""
    E = body.embed_tokens
    pre_ids = PREFIX_IDS if pre_ids is None else pre_ids
    pre = E(pre_ids.to(EMB_DEVICE))
    parts = [pre, vis.to(EMB_DEVICE, pre.dtype)] + ([NEWLINE.to(EMB_DEVICE, pre.dtype)[None]] if newline else [])
    x = torch.cat(parts, 0)[None]
    n_pre = pre.shape[0]
    vp = torch.as_tensor(vis_pos, device=EMB_DEVICE, dtype=torch.long)
    pos = torch.cat([torch.arange(n_pre, device=EMB_DEVICE), n_pre + vp] +
                    ([torch.tensor([n_pre + n_video_full], device=EMB_DEVICE)] if newline else []))[None]
    am = torch.ones(1, x.shape[1], dtype=torch.long, device=EMB_DEVICE)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    cache = _new_cache()
    _sync(); t0 = time.time()
    with sdpa_kernel(NO_MATH):
        body(inputs_embeds=x, attention_mask=am, position_ids=pos, past_key_values=cache, use_cache=True)
    _sync()
    return {"cache": cache, "n": int(x.shape[1]), "next_pos": int(n_pre + n_video_full + (1 if newline else 0)),
            "sec": time.time() - t0, "nvis": int(vis.shape[0]),
            "peak": max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9}

@torch.no_grad()
def suffix_pass(P, suf_ids, n_opt):
    """Question tokens on top of a cached prefix; the cache is cropped back afterwards so it can be reused.
    'sec' and 'flops' are reported for the pass as if run alone (prefix + suffix)."""
    E = body.embed_tokens
    suf = E(suf_ids.to(EMB_DEVICE))[None]
    n_q = suf.shape[1]
    pos = (P["next_pos"] + torch.arange(n_q, device=EMB_DEVICE))[None]
    am = torch.ones(1, P["n"] + n_q, dtype=torch.long, device=EMB_DEVICE)
    _sync(); t0 = time.time()
    # short query block over a long cache: explicit mask, small -> the plain math kernel is safe and cheap here
    with sdpa_kernel([SDPBackend.MATH]):
        out = body(inputs_embeds=suf, attention_mask=am, position_ids=pos, past_key_values=P["cache"], use_cache=True)
    h = out[0][0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    _crop(P["cache"], P["n"])
    lp = torch.log_softmax(logits[LETTER_IDS[:n_opt]], -1).cpu().numpy()
    if not np.all(np.isfinite(lp)):
        raise FloatingPointError("NaN/inf logits (fp16 overflow)")
    srt = np.sort(lp)[::-1]
    return {"pred": LETTERS[int(lp.argmax())], "logp": lp.round(4).tolist(), "margin": float(srt[0] - srt[1]),
            "ntok": P["nvis"], "vis_mean_layers": float(P["nvis"]), "sec": float(P["sec"] + dt),
            "sec_suffix": float(dt), "flops": llm_flops(P["n"] + n_q), "peak_gb": round(P["peak"], 3)}

@torch.no_grad()
def preflight_attention(n_tok):
    dev = body.layers[0].self_attn.q_proj.weight.device
    q = torch.randn(1, NHEADS, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    kv = torch.randn(1, NKV, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    mask = torch.ones(n_tok, n_tok, dtype=torch.bool, device=dev).tril()[None, None]
    torch.cuda.reset_peak_memory_stats(dev)
    base = torch.cuda.memory_allocated(dev)
    try:
        with sdpa_kernel(NO_MATH):
            o1 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, is_causal=True, enable_gqa=True)
            o2 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, attn_mask=mask, enable_gqa=True)  # routed to is_causal
        torch.cuda.synchronize(dev)          # surface any asynchronous CUDA error HERE, not in a later cell
        assert torch.equal(o1, o2), "causal-mask routing changed the result"
    except RuntimeError as e:
        raise RuntimeError(f"Memory-efficient attention is not available here ({e}). Send me this message.")
    del o1, o2
    peak = (torch.cuda.max_memory_allocated(dev) - base) / 1e9
    del q, kv, mask
    torch.cuda.empty_cache()
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU{i}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
    print(f"Attention preflight OK ✅  {n_tok} tokens -> extra peak memory {peak:.2f} GB")

preflight_attention(NUM_FRAMES * POOLED ** 2 + 400)
print("Model ready ✅")

### 4b · Safety checks (run on the first video)

In [ ]:
# ── Safety check: our token pipeline must reproduce the official model forward ──
from transformers import AutoProcessor

def equivalence_check(frames, row):
    report = {"transformers": transformers.__version__}
    try:
        proc = AutoProcessor.from_pretrained(MODEL_ID)
        opts = "\n".join(str(o) for o in row.options)
        text = ("Select the best answer to the following multiple-choice question based on the video. "
                "Respond with only the letter (A, B, C, or D) of the correct option.\n"
                f"{row.question}\n{opts}\nThe best answer is:")
        conv = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": text}]}]
        prompt = proc.apply_chat_template(conv, add_generation_prompt=True)
        inputs = proc(text=prompt, videos=[np.stack(frames)], return_tensors="pt")
        inputs = {k: (v.to(EMB_DEVICE, torch.float16) if torch.is_floating_point(v) else v.to(EMB_DEVICE))
                  for k, v in inputs.items()}
        n_opt = min(len(row.options), len(LETTERS))
        with torch.no_grad(), sdpa_kernel(NO_MATH):
            try:
                out = model(**inputs, logits_to_keep=1)
            except TypeError:
                out = model(**inputs, num_logits_to_keep=1)
        lp_off = torch.log_softmax(out.logits[0, -1].float()[LETTER_IDS[:n_opt]], -1).cpu().numpy()
        del out
        ids = inputs["input_ids"][0].cpu()
        vid_tok = getattr(CFG, "video_token_id", None) or getattr(CFG, "video_token_index", None)
        posi = (ids == vid_tok).nonzero().flatten()
        X = encode_video(inputs["pixel_values_videos"][0])[0]
        vis = X.reshape(-1, X.shape[-1])
        report["n_video_tokens_official"] = int(len(posi))
        report["n_video_tokens_ours"] = int(vis.shape[0] + 1)
        ours = llm_pass(vis, np.arange(vis.shape[0]), vis.shape[0], ids[posi[-1] + 1:], n_opt, pre_ids=ids[: posi[0]])
        lp_ours = np.array(ours["logp"])
        report["logp_official"] = lp_off.round(4).tolist()
        report["logp_ours_same_pixels"] = lp_ours.round(4).tolist()
        report["max_abs_gap"] = float(np.abs(lp_off - lp_ours).max())
        # our own decoder loop (used by FlashVID / HoliTom) must give the same answer as the library forward
        man = llm_pass(vis, np.arange(vis.shape[0]), vis.shape[0], ids[posi[-1] + 1:], n_opt, pre_ids=ids[: posi[0]],
                       manual=True)
        report["logp_ours_manual_loop"] = man["logp"]
        report["max_abs_gap_manual_loop"] = float(np.abs(np.array(man["logp"]) - lp_ours).max())
        report["pixel_mean_abs_diff_ours_vs_official"] = float(
            (preprocess(frames) - inputs["pixel_values_videos"][0].float().cpu()).abs().mean())
        ok = (report["n_video_tokens_official"] == report["n_video_tokens_ours"] and report["max_abs_gap"] <= 0.15
              and report["max_abs_gap_manual_loop"] <= 0.05)
        report["status"] = "PASS" if ok else "FAIL"
    except Exception as e:
        report["status"] = "SKIPPED"
        report["error"] = f"{type(e).__name__}: {e}"
    with open(os.path.join(OUT_DIR, "equivalence_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Equivalence check:", json.dumps(report, indent=2))
    if report["status"] == "FAIL" or (report["status"] == "SKIPPED" and not QUICK_TEST):
        raise RuntimeError("Equivalence check failed/skipped — stopping so no GPU hours are wasted. "
                           "Send me equivalence_check.json and the log.")
    gc.collect(); torch.cuda.empty_cache()


# ── Safety check 2: prefix caching must give the same answers as running each pass in full ──
def cache_check(Xf, comp, rows):
    """On the first video: full pass, option-reversed full pass and two compressed passes, cached vs uncached."""
    T, L, D = Xf.shape
    report = {"checks": []}
    picks = [("full", None)] + [k for k in comp if k in (("random", "0.1"), ("fastvid", "0.1"))]
    for key in picks:
        if key[0] == "full":
            vis, pos, nfull, nl = Xf.reshape(-1, D), np.arange(T * L), T * L, True
        else:
            c = comp[key]
            vis, pos, nfull, nl = c["tokens"], c["pos"], c["n_full"], c["newline"]
        P = prefill(vis, pos, nfull, newline=nl)
        for row in rows[:2]:
            n_opt = min(len(row.options), len(LETTERS))
            for opts in (row.options, reorder_options(row.options, "option_reverse")):
                suf = build_suffix_ids(row.question, opts)
                a = llm_pass(vis, pos, nfull, suf, n_opt, newline=nl)
                b = suffix_pass(P, suf, n_opt)
                report["checks"].append({"pass": "/".join(map(str, key)) if key[0] != "full" else "full",
                                         "qid": row.qid, "same_answer": a["pred"] == b["pred"],
                                         "gap": float(np.abs(np.array(a["logp"]) - np.array(b["logp"])).max()),
                                         "speedup_per_extra_question": round(a["sec"] / max(b["sec_suffix"], 1e-6), 1)})
        del P
    same = np.mean([c["same_answer"] for c in report["checks"]])
    report["max_gap"] = max(c["gap"] for c in report["checks"])
    report["same_answer_share"] = float(same)
    # pre-registered stop rule is about answers; the log-prob tolerance matches the equivalence check
    report["status"] = "PASS" if (same == 1.0 and report["max_gap"] <= 0.15) else "FAIL"
    with open(os.path.join(OUT_DIR, "cache_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Cache check:", report["status"], f"| max log-prob gap {report['max_gap']:.4f} | "
          f"same answers {100 * same:.0f}% | an extra question on a cached prefix is "
          f"~{np.median([c['speedup_per_extra_question'] for c in report['checks']]):.0f}x faster than a full pass")
    if report["status"] != "PASS":
        raise RuntimeError("Cache check failed — stopping so no GPU hours are wasted. Send me cache_check.json.")
    gc.collect(); torch.cuda.empty_cache()

### 5 · Compressors: all eight methods (+ self-test)

In [ ]:
# ══ Compressors, part 1 (Step 1 ports): FastVID, DyCoke, VisionZip, uniform pooling, random drop ══
# ── FastVID (Shen et al., NeurIPS 2025) — faithful port of the official LLaVA-OneVision code ──
# Source: github.com/LunarShen/FastVID, fastvid_llavaonevision/LLaVA-NeXT/llava/model/language_model/modeling_qwen2.py
# (MIT License, Copyright (c) 2025 Shen Leqi). Logic is unchanged; the only edits are
# (1) a seeded generator for the 1e-6 tie-break noise (reproducibility) and
# (2) the token-merging sums are done in fp32 instead of fp16 to avoid overflow on long segments, and
# (3) broadcasting matmul instead of an expanded bmm (same numbers, far less memory).
@torch.no_grad()
def fastvid_compress(X, attn, gfeat, r, c=8, tau=0.9, d=0.4, p=4, beta=0.6, seed=0):
    """X: (T, L, D) projected video tokens; attn: (T, L) SigLIP-head attention; gfeat: (T, Dg) pooled features.
    Returns kept tokens (K, D) and their original indices in [0, T*L), sorted."""
    T, L, D = X.shape
    dev = X.device
    gen = torch.Generator(device=dev).manual_seed(seed)
    batchframe_indices = torch.arange(T, device=dev).unsqueeze(1)
    alltoken_indices = torch.arange(T * L, device=dev).view(T, L)
    attn = attn.to(dev)
    gfeat = gfeat.to(dev)
    # ── DySeg ──
    g = gfeat / gfeat.norm(dim=1, keepdim=True)
    sim = (g[:-1] * g[1:]).sum(dim=1)
    cut_topk = torch.topk(sim, min(c - 1, T - 1), largest=False).indices
    cut_cos = torch.nonzero(sim < tau, as_tuple=False).squeeze(1)
    cut = torch.unique(torch.cat([cut_topk, cut_cos])).sort().values
    padded = F.pad(cut, (1, 1), value=-1)
    padded[-1] = T - 1
    segment_sizes = padded.diff().tolist()
    # ── STPrune ──
    final_tokens, keep = [], []
    frame_retain_num = int(L * r)
    frame_salient_num = frame_retain_num - int(frame_retain_num * d)
    frame_context_num = frame_retain_num - frame_salient_num
    frm_context_num_list = torch.zeros(T, dtype=torch.int, device=dev)
    offset = 0
    for seg_len in segment_sizes:
        temp_num = (seg_len + p - 1) // p
        cur = (frame_context_num * seg_len) // temp_num
        seg_idx = torch.arange(seg_len - 1, -1, -1, device=dev)
        sub = frm_context_num_list[offset:offset + seg_len]
        sub[seg_idx % p == 0] = cur
        offset += seg_len
    # ── ATS: attention-based salient tokens ──
    salient_indexes = torch.topk(attn, frame_salient_num, dim=1).indices
    bi = batchframe_indices.expand(-1, frame_salient_num)
    final_tokens.append(X[bi, salient_indexes].reshape(-1, D))
    keep.append(alltoken_indices[bi, salient_indexes].reshape(-1))
    # ── density scores on the remaining tokens ──
    all_indices = torch.arange(L, device=dev).unsqueeze(0).expand(T, -1)
    m = torch.ones_like(all_indices, dtype=torch.bool)
    m.scatter_(1, salient_indexes, False)
    nf = L - frame_salient_num
    filtered_indices = all_indices[m].view(T, nf)
    bi = batchframe_indices.expand(-1, nf)
    token_filtered = X[bi, filtered_indices]
    all_filtered_idx = alltoken_indices[bi, filtered_indices]
    dist = torch.cdist(token_filtered.float(), token_filtered.float()) / (D ** 0.5)
    dist_nearest, _ = torch.topk(dist, k=4, dim=-1, largest=False)
    density = (-(dist_nearest ** 2).mean(dim=-1)).exp()
    density = density + torch.rand(density.shape, device=dev, dtype=density.dtype, generator=gen) * 1e-6
    dmask = (density[:, None, :] > density[:, :, None]).float()
    dist_max = dist.flatten(1).max(dim=-1)[0][:, None, None]
    dist_0, _ = (dist * dmask + dist_max * (1 - dmask)).min(dim=-1)
    density_score = dist_0 * density
    sampled = torch.topk(density_score, k=frame_context_num, dim=-1).indices
    tf32 = token_filtered.float()
    # ── DTM for single-frame segments ──
    bi = batchframe_indices.expand(-1, frame_context_num)
    ctx = tf32[bi, sampled]
    ctx_idx = all_filtered_idx[bi, sampled]
    tbm = tf32 / tf32.norm(dim=-1, keepdim=True)
    mt = tbm[bi, sampled]
    simm = torch.bmm(tbm, mt.transpose(1, 2))
    a1 = torch.zeros(T, nf, frame_context_num, device=dev)
    a1.scatter_(2, simm.argmax(dim=2).unsqueeze(-1), 1)
    aw = (1 / (a1.sum(dim=1).unsqueeze(-1) + 1)).clamp(min=beta)
    cnt = a1.sum(dim=1).clamp(min=1).unsqueeze(-1)
    agg = torch.bmm(a1.transpose(1, 2), tf32) / cnt
    ctx = aw * ctx + (1 - aw) * agg
    single = frm_context_num_list == frame_context_num
    final_tokens.append(ctx[single].reshape(-1, D).to(X.dtype))
    keep.append(ctx_idx[single].reshape(-1))
    # ── DTM for multi-frame segments ──
    start = 0
    for seg_len in segment_sizes:
        if seg_len > 1:
            lst = frm_context_num_list[start:start + seg_len]
            k_ctx = int(lst[-1])
            tmask = lst > frame_context_num
            tnum = int(tmask.sum())
            seg_ds = density_score[start:start + seg_len][tmask]
            seg_tok = tf32[start:start + seg_len]
            seg_target = seg_tok[tmask]
            seg_flat = seg_tok.reshape(-1, D)                    # all filtered tokens of the segment (shared by every anchor)
            seg_idx = all_filtered_idx[start:start + seg_len][tmask]
            smp = torch.topk(seg_ds, k=k_ctx, dim=-1).indices
            bj = batchframe_indices[:tnum].expand(-1, k_ctx)
            cctx = seg_target[bj, smp]
            cidx = seg_idx[bj, smp]
            tb = seg_flat / seg_flat.norm(dim=-1, keepdim=True)
            mtt = cctx / cctx.norm(dim=-1, keepdim=True)
            s2 = torch.matmul(tb[None], mtt.transpose(1, 2))      # = official bmm with the expanded segment
            a2 = torch.zeros(tnum, tb.shape[0], k_ctx, device=dev)
            a2.scatter_(2, s2.argmax(dim=2).unsqueeze(-1), 1)
            aw2 = (1 / (a2.sum(dim=1).unsqueeze(-1) + 1)).clamp(min=beta)
            cnt2 = a2.sum(dim=1).clamp(min=1).unsqueeze(-1)
            agg2 = torch.matmul(a2.transpose(1, 2), seg_flat) / cnt2
            cctx = aw2 * cctx + (1 - aw2) * agg2
            final_tokens.append(cctx.reshape(-1, D).to(X.dtype))
            keep.append(cidx.reshape(-1))
        start += seg_len
    toks = torch.cat(final_tokens, 0)
    idx = torch.cat(keep, 0)
    order = torch.argsort(idx)
    return toks[order], idx[order]


# ── DyCoke (Tao et al., CVPR 2025) — temporal token merging (TTM), faithful port ──
# Source: github.com/KD-TAO/DyCoke, llava/model/llava_arch.py `dycole_ttm` (Apache-2.0). Applied, as in the official
# code, to the flattened pooled projected tokens; kept tokens get contiguous positions. Stage 2 (KV-cache pruning)
# only runs at decoding steps (position_ids length 1), so it does not affect a one-token answer.
# The official loop structure is kept exactly, including that frame 30 of 32 stays unpruned.
@torch.no_grad()
def dycoke_ttm(X, K):
    T, L, D = X.shape
    feat = X.reshape(T * L, D)
    keep_frac = 1 - K
    n_keep = int(keep_frac * L)
    fr = lambda j: feat[j * L:(j + 1) * L]
    sims = []
    for i in range(0, T - 1, 2):
        a = F.normalize(fr(i).float(), p=2, dim=1); b = F.normalize(fr(i + 1).float(), p=2, dim=1)
        sims.append(F.cosine_similarity(a, b, dim=1))
    out = []
    for i in range(0, T - 1, 2):
        keep = sims[i // 2].topk(n_keep, largest=False).indices
        out.append(fr(i)); out.append(fr(i + 1)[keep])
    odd = []
    for i in range(0, T - 4, 4):
        odd.append(F.cosine_similarity(fr(i).float(), fr(i + 2).float(), dim=1))
    for i in range(0, T - 4, 4):
        keep = odd[i // 4].topk(n_keep, largest=False).indices
        out[i] = fr(i); out[i + 2] = fr(i + 2)[keep]
    return torch.cat(out, 0)

# ── VisionZip (Yang et al., CVPR 2025) — LLaVA-OV video port from the HoliTom repo (visionzip/llava_arch.py) ──
# Per frame, after LLaVA-OV's 2x2 pooling: dominant tokens by received vision attention, contextual tokens by merging
# the rest onto evenly spaced targets using key similarity. Default dominant ratio 54/64 (as in that code).
# Output order per frame = [dominant (spatial order), contextual]; frames concatenated; contiguous positions.
@torch.no_grad()
def visionzip_compress(X, vz_attn, vz_metric, r, dominant_ratio=54 / 64):
    T, S, D = X.shape
    spatial = int(S * r)
    dom = int(spatial * dominant_ratio)
    ctx = spatial - dom
    hs = X.float()
    att = vz_attn.to(hs.device).float()
    met = vz_metric.to(hs.device).float()
    topk = att.topk(dom, dim=1).indices
    mask = torch.ones(T, S, dtype=torch.bool, device=hs.device).scatter_(1, topk, False)   # True = not dominant
    dominant = hs.masked_select(~mask.unsqueeze(-1)).view(T, dom, D)
    met_f = met[mask].view(T, S - dom, met.shape[2])
    hs_f = hs.masked_select(mask.unsqueeze(-1)).view(T, S - dom, D)
    met_n = met_f / met_f.norm(dim=-1, keepdim=True)
    step = max(1, (S - dom) // ctx)
    tgt = torch.arange(0, S - dom, step, device=hs.device)[:ctx]
    tgt_tok = met_n[:, tgt, :]
    rest = ~torch.isin(torch.arange(met_n.shape[1], device=hs.device), tgt)
    to_merge = met_n[:, rest, :]
    sim = torch.bmm(to_merge, tgt_tok.transpose(1, 2))
    onehot = torch.zeros(T, to_merge.shape[1], ctx, device=hs.device)
    onehot.scatter_(2, sim.argmax(dim=2).unsqueeze(-1), 1)
    counts = onehot.sum(dim=1).clamp(min=1).unsqueeze(-1)
    agg = torch.bmm(onehot.transpose(1, 2), hs_f[:, rest, :]) / counts
    contextual = hs_f[:, tgt, :] + agg
    return torch.cat([dominant, contextual], dim=1).reshape(-1, D).to(X.dtype)

# ── Baselines ──
def _best_grid(n, side):
    best = None
    for h in range(1, side + 1):
        for w in range(1, side + 1):
            key = (abs(h * w - n), abs(h - w), -h * w)
            if best is None or key < best[0]:
                best = (key, h, w)
    return best[1], best[2]

@torch.no_grad()
def uniform_pool(X, r):
    """Per-frame average pooling of the 14x14 grid to the grid closest to int(196*r) tokens."""
    T, S, D = X.shape
    side = int(round(S ** 0.5))
    h, w = _best_grid(int(S * r), side)
    g = X.float().view(T, side, side, D).permute(0, 3, 1, 2)
    g = F.adaptive_avg_pool2d(g, (h, w))
    return g.permute(0, 2, 3, 1).reshape(-1, D).to(X.dtype)

@torch.no_grad()
def random_drop(X, r, seed):
    """Per frame, keep int(196*r) uniformly random tokens in spatial order."""
    T, S, D = X.shape
    k = int(S * r)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.stack([torch.randperm(S, generator=gen)[:k].sort().values for _ in range(T)]).to(X.device)
    return torch.gather(X, 1, idx.unsqueeze(-1).expand(-1, -1, D)).reshape(-1, D)



# ══ Compressors, part 2 (Step 1b ports): FlashVID, HoliTom, LLaVA-Scissor ══
# Each compressor returns a dict: tokens (K, D) entering the LLM, pos (their position ids, contiguous for all three
# as in the official code), newline (whether LLaVA-OV's video newline token is appended), and inner (an optional
# inner-LLM operation applied by llm_pass at a given layer).

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# FlashVID (Fan et al., ICLR 2026 oral) — github.com/Fanziyang-v/FlashVID, flashvid/utils.py + token_selection.py
# MIT License, Copyright (c) the FlashVID authors. The functions below are copied verbatim (only the config class is
# re-declared here); the official LLaVA-OV path passes the bilinear-pooled features and the bilinear-pooled
# last-vision-layer attention (mean over heads and queries), then prunes inside the LLM at layer 20 (FastV-style,
# last-query attention of layer 19, keep 30%). Positions are contiguous before the LLM, original after pruning.
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Optional, Tuple, Union, List

@dataclass
class FlashVidConfig:
    retention_ratio: float = 0.25
    alpha: float = 0.7
    token_selection_method: str = "attn_div_v2"
    temporal_threshold: float = 0.8
    do_segment: bool = True
    segment_threshold: float = 0.9
    min_segment_num: int = 8
    complementary_segment: bool = True
    num_attn_div_tokens: Optional[int] = None
    num_sttm_tokens: Optional[int] = None
    visual_token_start_index: Optional[int] = None
    visual_token_length: Optional[int] = None
    expansion: float = 1.25
    pruning_layer: int = 20
    llm_retention_ratio: float = 0.3

def pairwise_cosine_distances(image_features):
    normed_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    similarities = torch.bmm(normed_features, normed_features.transpose(-1, -2))
    return 1.0 - similarities

def attn_div_v2_based_token_selection(features, cls_attention, num_retained_tokens):
    original_features = features
    features = features.float()
    pooled_features = features.mean(1)
    global_cls_attention = cls_attention.float() * 1e6
    bsz, num_visual_tokens, feat_dim = features.shape
    dist_matrix = pairwise_cosine_distances(features)
    calibration_term1 = global_cls_attention.unsqueeze(1)
    local_cls_attention = torch.einsum("b n d, c d -> b c n", features, pooled_features).mean(1)
    calibration_term2 = local_cls_attention.unsqueeze(1)
    dist_matrix = dist_matrix * calibration_term1 * calibration_term2
    keep_indices = torch.zeros(bsz, num_retained_tokens, dtype=torch.long, device=features.device)
    min_dist = torch.topk(dist_matrix, k=2, dim=1, largest=False).values[:, 1, :]
    keep_indices[:, 0] = torch.argmax(min_dist, dim=-1)
    for i in range(1, num_retained_tokens):
        dist_sub_matrix = torch.gather(dist_matrix, dim=1,
                                       index=keep_indices[:, :i].unsqueeze(-1).expand(-1, -1, num_visual_tokens))
        min_dist = torch.min(dist_sub_matrix, dim=1).values
        keep_indices[:, i] = torch.argmax(min_dist, dim=-1)
    keep_indices = keep_indices.sort().values
    selected_features = torch.gather(original_features, dim=1,
                                     index=keep_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, keep_indices

def attn_based_token_selection(features, cls_attention, num_retained_tokens):
    bsz, num_visual_tokens, feat_dim = features.shape
    topk_indices = torch.topk(cls_attention, k=num_retained_tokens, dim=-1).indices.sort().values
    selected_features = torch.gather(features, dim=1, index=topk_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, topk_indices

ALL_TOKEN_SELECTION_METHOD = {"attn_div_v2": attn_div_v2_based_token_selection, "attn": attn_based_token_selection}

def flashvid_compression(video_features, cls_attention, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = video_features.shape
    if flashvid_config.do_segment:
        segment_lengths = fv_segment(video_features=video_features.mean(1),
                                     segment_threshold=flashvid_config.segment_threshold,
                                     min_segment_num=flashvid_config.min_segment_num,
                                     complementary_segment=flashvid_config.complementary_segment)
    else:
        segment_lengths = torch.tensor([num_frames], dtype=torch.long, device=video_features.device)
    num_segments = segment_lengths.shape[0]
    global_indices = torch.arange(num_frames * num_visual_tokens, dtype=torch.long, device=video_features.device)
    token_budget = math.ceil(num_visual_tokens * flashvid_config.retention_ratio * flashvid_config.expansion)
    num_attn_div_tokens = math.ceil(token_budget * flashvid_config.alpha)
    num_sttm_tokens = token_budget - num_attn_div_tokens
    flashvid_config.num_attn_div_tokens = num_attn_div_tokens
    flashvid_config.num_sttm_tokens = num_sttm_tokens
    all_segment_features, all_segment_indices = [], []
    offset = 0
    for seg_idx in range(num_segments):
        seg_len = segment_lengths[seg_idx]
        segment_features = video_features[offset: offset + seg_len]
        segment_cls_attention = cls_attention[offset: offset + seg_len]
        segment_global_indices = global_indices.view(num_frames, num_visual_tokens)[offset: offset + seg_len]
        segment_features, segment_global_indices = fv_segment_compression(
            segment_features=segment_features, segment_global_indices=segment_global_indices,
            cls_attention=segment_cls_attention, flashvid_config=flashvid_config)
        all_segment_features.append(segment_features)
        all_segment_indices.append(segment_global_indices)
        offset += seg_len
    final_tokens = torch.cat(all_segment_features, dim=0)
    final_global_indices = torch.cat(all_segment_indices, dim=0)
    sorted_indices = final_global_indices.argsort()
    sorted_tokens = final_tokens[sorted_indices]
    flashvid_config.visual_token_length = sorted_tokens.shape[0]
    return sorted_tokens, final_global_indices[sorted_indices]

def fv_segment_compression(segment_features, segment_global_indices, cls_attention, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = segment_features.shape
    if flashvid_config.alpha > 0:
        additional_kwargs = {"cls_attention": cls_attention} if "attn" in flashvid_config.token_selection_method else {}
        selected_features, selected_indices = ALL_TOKEN_SELECTION_METHOD[flashvid_config.token_selection_method](
            features=segment_features, num_retained_tokens=flashvid_config.num_attn_div_tokens, **additional_kwargs)
        selected_global_indices = segment_global_indices.gather(1, index=selected_indices).view(-1)
    else:
        selected_features = torch.tensor([]).to(segment_features)
        selected_indices = torch.tensor([]).to(segment_global_indices)
        selected_global_indices = torch.tensor([]).to(segment_global_indices)
    mask = torch.ones(num_frames, num_visual_tokens, dtype=torch.bool, device=segment_features.device)
    mask.scatter_(1, selected_indices, False)
    num_other_tokens = flashvid_config.num_sttm_tokens * num_frames
    if num_other_tokens > 0 and flashvid_config.temporal_threshold < 1.0:
        if num_frames > 1:
            temp_merged_token_list, temp_merged_indices_list = fv_spatiotemporal_compression(
                video_features=segment_features, temporal_threshold=flashvid_config.temporal_threshold,
                token_mask=mask, flashvid_config=flashvid_config)
            temp_merged_global_indices_list = [segment_global_indices.view(num_frames, -1)[i][temp_merged_indices]
                                               for i, temp_merged_indices in enumerate(temp_merged_indices_list)]
        else:
            temp_merged_token_list = [segment_features[0]]
            temp_merged_global_indices_list = [segment_global_indices[0]]
    else:
        temp_merged_token_list = []
        temp_merged_global_indices_list = []
    all_tokens = [selected_features.view(-1, feat_dim)]
    all_global_indices = [selected_global_indices]
    if num_other_tokens > 0:
        num_current_retained_tokens = sum(len(tokens) for tokens in temp_merged_token_list)
        adapative_contextual_ratio = num_other_tokens / num_current_retained_tokens
        if adapative_contextual_ratio < 1.0:
            num_frames_in_segment = len(temp_merged_token_list)
            max_num_tokens = max(len(tokens) for tokens in temp_merged_token_list)
            batched_tokens = torch.zeros((num_frames_in_segment, max_num_tokens, feat_dim),
                                         dtype=segment_features.dtype, device=segment_features.device)
            valid_token_mask = torch.zeros((num_frames_in_segment, max_num_tokens), dtype=torch.bool,
                                           device=segment_features.device)
            num_clusters_list, k_list = [], []
            for i, temp_merged_tokens in enumerate(temp_merged_token_list):
                num_tokens = len(temp_merged_tokens)
                batched_tokens[i, :num_tokens] = temp_merged_tokens
                valid_token_mask[i, :num_tokens] = True
                num_clusters = math.ceil(num_tokens * adapative_contextual_ratio)
                num_clusters_list.append(num_clusters)
                k_list.append(min(num_clusters, 7))
            cluster_indices_list, cluster_center_indices_list = fv_dpc_knn(
                features=batched_tokens, num_clusters=num_clusters_list, k=k_list, valid_token_mask=valid_token_mask)
            for i, (temp_merged_tokens, temp_merged_global_indices) in enumerate(
                    zip(temp_merged_token_list, temp_merged_global_indices_list)):
                num_clusters = num_clusters_list[i]
                if num_clusters > 0:
                    cluster_indices = cluster_indices_list[i][:len(temp_merged_tokens)]
                    cluster_center_indices = cluster_center_indices_list[i]
                    aggregated_tokens = torch.zeros((num_clusters, feat_dim), dtype=segment_features.dtype,
                                                    device=segment_features.device)
                    aggregated_tokens.scatter_add_(0, cluster_indices.unsqueeze(-1).expand(-1, feat_dim),
                                                   temp_merged_tokens)
                    cluster_counts = torch.bincount(cluster_indices, minlength=num_clusters).unsqueeze(-1).to(
                        segment_features.dtype)
                    aggregated_tokens = aggregated_tokens / cluster_counts
                    global_token_indices = temp_merged_global_indices[cluster_center_indices]
                else:
                    aggregated_tokens = temp_merged_tokens
                    global_token_indices = temp_merged_global_indices
                all_tokens.append(aggregated_tokens)
                all_global_indices.append(global_token_indices)
        else:
            for temp_merged_tokens, temp_merged_global_indices in zip(temp_merged_token_list,
                                                                      temp_merged_global_indices_list):
                all_tokens.append(temp_merged_tokens)
                all_global_indices.append(temp_merged_global_indices)
    segment_final_tokens = torch.cat(all_tokens, dim=0)
    segment_final_global_indices = torch.cat(all_global_indices, dim=0)
    return segment_final_tokens, segment_final_global_indices

def fv_segment(video_features, segment_threshold, min_segment_num, complementary_segment=True):
    num_frames, feat_dim = video_features.shape
    normed_video_features = video_features / video_features.norm(p=2, dim=-1, keepdim=True)
    transition_similarities = torch.sum(normed_video_features[:-1] * normed_video_features[1:], dim=-1)
    cut_indices = torch.where(transition_similarities < segment_threshold)[0]
    return fv_additional_segment(cut_indices=cut_indices, num_frames=num_frames, min_segment_num=min_segment_num,
                                 transition_similarities=transition_similarities,
                                 segment_threshold=segment_threshold, complementary_segment=complementary_segment)

def fv_additional_segment(cut_indices, num_frames, min_segment_num, transition_similarities, segment_threshold,
                          complementary_segment=True):
    num_segments = cut_indices.numel() + 1
    if num_segments < min_segment_num and complementary_segment:
        num_remaining_cut_indices = min_segment_num - num_segments
        transition_similarities[transition_similarities < segment_threshold] = 1.0
        complementary_cut_indices = torch.topk(transition_similarities,
                                               k=min(num_remaining_cut_indices, transition_similarities.shape[0]),
                                               largest=False).indices
        cut_indices = torch.cat([cut_indices, complementary_cut_indices]).sort().values
    padded_cut_indices = F.pad(cut_indices, (1, 1), value=0)
    padded_cut_indices[0] = -1
    padded_cut_indices[-1] = num_frames - 1
    return torch.diff(padded_cut_indices, n=1, dim=0)

@torch.no_grad()
def fv_dpc_knn(features, num_clusters, k=7, valid_token_mask=None):
    invalid_token_mask = ~valid_token_mask if valid_token_mask is not None else None
    bsz, seq_len, feat_dim = features.shape
    dists = torch.cdist(features.float(), features.float()) / math.sqrt(feat_dim)
    if valid_token_mask is not None:
        dists = torch.masked_fill(dists, invalid_token_mask.unsqueeze(1).expand(-1, seq_len, -1), dists.max() + 1)
    max_k = max(k) if isinstance(k, list) else k
    nearest_dist = torch.topk(dists, k=max_k, dim=-1, largest=False).values
    if isinstance(k, list):
        density = torch.empty((bsz, seq_len), device=features.device, dtype=features.dtype)
        for i in range(bsz):
            density[i] = torch.mean(-(nearest_dist[i, :, :k[i]] ** 2), dim=-1).exp()
    else:
        density = torch.mean(-(nearest_dist ** 2), dim=-1).exp()
    if valid_token_mask is not None:
        density = torch.masked_fill(density, invalid_token_mask, 0.0)
    mask = density[:, None, :] > density[:, :, None]
    max_dist = dists.view(bsz, -1).max(dim=-1)[0].view(-1, 1, 1)
    modified_dists = torch.where(mask, dists, max_dist)
    dist, _ = torch.min(modified_dists, dim=-1)
    score = dist * density
    if isinstance(num_clusters, int):
        cluster_center_indices = torch.topk(score, k=num_clusters, dim=-1).indices
        dists = torch.gather(dists, dim=-1, index=cluster_center_indices.unsqueeze(1).expand(-1, seq_len, -1))
        cluster_indices = torch.argmin(dists, dim=-1)
        cluster_indices.scatter_(dim=-1, index=cluster_center_indices,
                                 src=torch.arange(num_clusters).to(cluster_indices).unsqueeze(0).expand(bsz, -1))
        return cluster_indices, cluster_center_indices
    cluster_indices_list, cluster_center_indices_list = [], []
    for i in range(bsz):
        k_i = num_clusters[i]
        if k_i == 0:
            cluster_center_indices_list.append(torch.tensor([], dtype=torch.long, device=features.device))
            cluster_indices_list.append(torch.zeros(seq_len, dtype=torch.long, device=features.device))
            continue
        cc_idx = torch.topk(score[i], k=k_i, dim=-1).indices
        cluster_center_indices_list.append(cc_idx)
        dists_i = torch.gather(dists[i], dim=-1, index=cc_idx.unsqueeze(0).expand(seq_len, -1))
        c_idx = torch.argmin(dists_i, dim=-1)
        c_idx.scatter_(dim=-1, index=cc_idx, src=torch.arange(k_i).to(c_idx))
        cluster_indices_list.append(c_idx)
    return cluster_indices_list, cluster_center_indices_list

def fv_spatiotemporal_compression(video_features, temporal_threshold, token_mask, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = video_features.shape
    lower_bound = (flashvid_config.num_attn_div_tokens + flashvid_config.num_sttm_tokens) * num_frames
    normed_video_features = video_features / video_features.norm(p=2, dim=-1, keepdim=True)
    cosine_similarities = torch.bmm(normed_video_features[1:], normed_video_features[:-1].transpose(1, 2))
    cosine_similarities[~token_mask[1:].unsqueeze(-1).expand(-1, -1, num_visual_tokens)] = -1.0
    cosine_similarities[~token_mask[:-1].unsqueeze(1).expand(-1, num_visual_tokens, -1)] = -1.0
    max_sims, max_sim_indices = torch.max(cosine_similarities, dim=-1)
    padded_max_sims = F.pad(max_sims, (0, 0, 1, 0), value=-1)
    padded_max_sim_indices = F.pad(max_sim_indices, (0, 0, 1, 0), value=-1)
    token_counts = torch.ones(num_frames, num_visual_tokens).to(video_features)
    mask = padded_max_sims > temporal_threshold
    retaining_token_mask = ~mask
    if retaining_token_mask.int().sum() < lower_bound:
        soft_threshold = padded_max_sims.view(-1).topk(k=(num_frames * num_visual_tokens) - lower_bound).values[-1]
        soft_threshold = max(soft_threshold, -1.0 + 1e-6)
        mask = padded_max_sims > soft_threshold
        retaining_token_mask = ~mask
    for frame_idx in range(num_frames - 1, -1, -1):
        frame_features = video_features[frame_idx]
        frame_token_counts = token_counts[frame_idx]
        frame_max_sim_indices = padded_max_sim_indices[frame_idx]
        tokens_to_merge = frame_features[~mask[frame_idx]]
        to_merge_token_counts = frame_token_counts[~mask[frame_idx]]
        if tokens_to_merge.numel() > 0:
            aggregated_tokens = tokens_to_merge / to_merge_token_counts.unsqueeze(-1).to(tokens_to_merge.dtype)
            video_features[frame_idx][~mask[frame_idx]] = aggregated_tokens
            token_counts[frame_idx][~mask[frame_idx]] = 1
        other_tokens = frame_features[mask[frame_idx]]
        if other_tokens.numel() > 0:
            anchor_token_indices = frame_max_sim_indices[mask[frame_idx]]
            aggregated_tokens = torch.zeros((num_visual_tokens, feat_dim), dtype=video_features.dtype,
                                            device=video_features.device)
            aggregated_tokens.scatter_add_(0, anchor_token_indices.unsqueeze(-1).expand(-1, feat_dim), other_tokens)
            aggregated_token_counts = torch.bincount(anchor_token_indices, minlength=num_visual_tokens).to(
                video_features.dtype)
            video_features[frame_idx - 1] += aggregated_tokens
            token_counts[frame_idx - 1] += aggregated_token_counts
            token_counts[frame_idx][mask[frame_idx]] = 0
    final_tokens, retained_token_indices = [], []
    for i in range(num_frames):
        frame_mask = retaining_token_mask[i] & token_mask[i]
        final_tokens.append(video_features[i][frame_mask])
        retained_token_indices.append(torch.where(frame_mask)[0])
    return final_tokens, retained_token_indices

@torch.no_grad()
def flashvid_compress(X, cls_attn, r):
    cfg = FlashVidConfig(retention_ratio=r, **FLASHVID)
    # official code edits the pooled features in place, so work on a copy; the attention enters in the model dtype
    toks, idx = flashvid_compression(X.clone(), cls_attn.to(X.device, X.dtype), cfg)
    inner = {"kind": "fastv", "layer": cfg.pruning_layer, "keep_ratio": cfg.llm_retention_ratio,
             "include_newline": False}
    return {"tokens": toks, "pos": np.arange(toks.shape[0]), "newline": True, "inner": inner}


# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# HoliTom (Shao et al., NeurIPS 2025) — github.com/cokeshao/HoliTom, holitom/llava_arch.py + modeling_qwen2.py
# Apache-2.0. Methods copied as plain functions (self removed); settings from scripts/eval_ov-7b_holitom.sh:
# T=0.65 at 10% (0.80 otherwise), BETA=0.6 (beta weighting on, as the script leaves NO_BETA unset), D=0, K=7,
# inner-LLM merging at layer 18 with r=0.5 (HOLITOM_k=18, HOLITOM_r=0.5). Only edit: the 1e-6 density tie-break noise
# uses a seeded generator. LLaVA-OV uses mm_newline_position "one_token", so tokens are [static, dynamic] per window,
# one newline at the end, contiguous positions; the inner merge treats the newline as part of the image span (as there).
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
def ht_cluster_dpc_knn(x, cluster_num, k=7, gen=None):
    with torch.no_grad():
        batch_size, seq_len, embed_dim = x.shape
        dist_matrix = torch.cdist(x.float(), x.float()) / (embed_dim ** 0.5)
        dist_nearest, index_nearest = torch.topk(dist_matrix, k, dim=-1, largest=False)
        density = (-(dist_nearest ** 2).mean(dim=-1)).exp()
        density = density + torch.rand(density.shape, device=density.device, dtype=density.dtype, generator=gen) * 1e-6
        mask = (density[:, None, :] > density[:, :, None]).type(x.dtype)
        dist_max = dist_matrix.flatten(1).max(dim=-1).values[:, None, None]
        dist, index_parent = (dist_matrix * mask + dist_max * (1 - mask)).min(dim=-1)
        score = dist * density
        _, index_center = score.topk(cluster_num, dim=-1)
        return index_center, dist_matrix

def ht_select_static_windows(feature_sim, batch_size, tau, max_window_size):
    def get_pruned_static_count_vectorized(feature_sim, batch_size, tau):
        similarity_matrix = torch.ones((batch_size, batch_size, feature_sim.shape[1]), device=feature_sim.device)
        for start in range(batch_size - 1):
            cum_similarity = torch.cumprod(feature_sim[start:] > tau, dim=0)
            similarity_matrix[start, start + 1:start + 1 + len(cum_similarity)] = cum_similarity
        window_lengths = torch.arange(batch_size, device=feature_sim.device).unsqueeze(0) - \
            torch.arange(batch_size, device=feature_sim.device).unsqueeze(1)
        window_lengths = window_lengths.clamp(min=0)
        return (similarity_matrix.sum(dim=-1) * window_lengths).float()
    pruned_static_count = get_pruned_static_count_vectorized(feature_sim, batch_size, tau)
    dp = torch.zeros(batch_size, device=pruned_static_count.device)
    prev = torch.zeros(batch_size, dtype=torch.long, device=pruned_static_count.device)
    for i in range(batch_size):
        max_val = dp[i - 1] if i > 0 else 0
        best_j = i
        for window_size in range(2, min(i + 1, max_window_size) + 1):
            j = i - window_size
            current_val = (dp[j] if j >= 0 else 0) + pruned_static_count[j + 1, i]
            if current_val > max_val:
                max_val = current_val
                best_j = j + 1
        dp[i] = max_val
        prev[i] = best_j
    selected_frames = []
    i = batch_size - 1
    while i >= 0:
        selected_frames.append((prev[i].item(), i))
        i = prev[i].item() - 1
    selected_frames = selected_frames[::-1]
    return selected_frames, dp[-1].item()

def ht_merge_tokens_by_clustering(feat, target_indices, dist_matrix, cluster_num, Beta):
    batch_size, seq_len, embed_dim = feat.shape
    all_indices = torch.arange(seq_len, device=feat.device).unsqueeze(0).expand(batch_size, -1)
    non_target_indices = torch.zeros((batch_size, seq_len - cluster_num), dtype=torch.long, device=feat.device)
    for b in range(batch_size):
        non_target_mask = ~torch.isin(all_indices[b], target_indices[b])
        non_target_indices[b] = all_indices[b][non_target_mask]
    non_target_feat = torch.gather(feat, dim=1, index=non_target_indices.unsqueeze(-1).expand(-1, -1, feat.size(-1)))
    dist_matrix = torch.gather(dist_matrix, dim=1,
                               index=non_target_indices.unsqueeze(-1).expand(-1, -1, dist_matrix.size(-1)))
    dist_matrix = torch.gather(dist_matrix, dim=2, index=target_indices.unsqueeze(1).expand(-1, dist_matrix.size(1), -1))
    idx_cluster = torch.argmin(dist_matrix, dim=-1)
    cluster_tokens = []
    for b in range(batch_size):
        batch_tokens = []
        for i in range(cluster_num):
            mask = (idx_cluster[b] == i)
            if mask.any():
                cluster_means = non_target_feat[b][mask].mean(dim=0)          # NO_BETA unset -> beta weighting
                batch_tokens.append(Beta * feat[b][target_indices[b][i]] + (1 - Beta) * cluster_means)
            else:
                batch_tokens.append(feat[b][target_indices[b][i]])
        cluster_tokens.append(torch.stack(batch_tokens))
    return torch.stack(cluster_tokens)

def ht_merge_tokens_by_attention_density(feat, attn, pos, retain_ratio, D, Beta, K, gen=None):
    batch_size, seq_len, embed_dim = feat.shape
    dominant_num = round(math.ceil(seq_len * retain_ratio) * (1 - D))
    contextual_num = math.ceil(seq_len * retain_ratio) - dominant_num
    if dominant_num > 0:
        all_indices = attn.topk(dominant_num, dim=1).indices
        mask = torch.ones_like(feat[:, :, 0], dtype=torch.bool, device=feat.device).scatter_(1, all_indices, False)
        dominant_tokens = feat.masked_select(~mask.unsqueeze(-1)).view(batch_size, dominant_num, embed_dim)
        dominant_pos = pos.masked_select(~mask).view(batch_size, dominant_num)
    else:
        mask = torch.ones_like(feat[:, :, 0], dtype=torch.bool, device=feat.device)
        dominant_tokens = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        dominant_pos = torch.empty((batch_size, 0), device=feat.device)
    if contextual_num > 0:
        feat_filtered = feat.masked_select(mask.unsqueeze(-1)).view(batch_size, seq_len - dominant_num, embed_dim)
        contextual_pos = pos.masked_select(mask.unsqueeze(-1)).view(batch_size, seq_len - dominant_num)
        target_indices, dist_matrix = ht_cluster_dpc_knn(feat_filtered, contextual_num, k=min(K, contextual_num), gen=gen)
        target_indices = torch.sort(target_indices, dim=-1)[0]
        contextual_pos = torch.stack([contextual_pos[b][target_indices[b]] for b in range(batch_size)])
        contextual_tokens = ht_merge_tokens_by_clustering(feat_filtered, target_indices, dist_matrix, contextual_num, Beta)
    else:
        contextual_tokens = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        contextual_pos = torch.empty((batch_size, 0), device=feat.device)
    image_feat, image_pos = [], []
    for b in range(batch_size):
        image_feat.append(torch.cat([dominant_tokens[b], contextual_tokens[b]], dim=0))
        image_pos.append(torch.cat([dominant_pos[b], contextual_pos[b]], dim=0))
    return torch.stack(image_feat), torch.stack(image_pos)

def ht_merge_tokens_by_density(feat, pos, retain_ratio, Beta, K, gen=None):
    batch_size, seq_len, embed_dim = feat.shape
    cluster_num = round(seq_len * retain_ratio)
    if cluster_num > 0:
        target_indices, dist_matrix = ht_cluster_dpc_knn(feat, cluster_num, k=min(K, cluster_num), gen=gen)
        target_indices = torch.sort(target_indices, dim=-1)[0]
        image_pos = torch.stack([pos[b][target_indices[b]] for b in range(batch_size)])
        image_feat = ht_merge_tokens_by_clustering(feat, target_indices, dist_matrix, cluster_num, Beta)
    else:
        image_feat = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        image_pos = torch.empty((batch_size, 0), device=feat.device)
    return image_feat, image_pos

def ht_holitom(static_feat, dynamic_feat, dynamic_attn, static_pos, dynamic_pos, window_size, retain_ratio, D, Beta,
               K, images_dtype, gen=None):
    # mm_newline_position == "one_token" branch (LLaVA-OneVision)
    if window_size == 1:
        dynamic_feat, dynamic_pos = ht_merge_tokens_by_attention_density(dynamic_feat, dynamic_attn, dynamic_pos,
                                                                         retain_ratio, D, Beta, K, gen)
        return dynamic_feat.flatten(0, 1).to(images_dtype)
    dynamic_feat, dynamic_pos = ht_merge_tokens_by_attention_density(dynamic_feat, dynamic_attn, dynamic_pos,
                                                                     retain_ratio, D, Beta, K, gen)
    static_feat, static_pos = ht_merge_tokens_by_density(static_feat.unsqueeze(0), static_pos, retain_ratio, Beta, K, gen)
    return torch.cat([static_feat.flatten(0, 1), dynamic_feat.flatten(0, 1)]).to(images_dtype)

def ht_get_static_dynamic_features(image_feat, attn_weights, selected_frames, feature_sim, tau):
    batch_size, seq_len, embed_dim = image_feat.shape
    static_feat_list, dynamic_feat_list, dynamic_attn_list, static_pos_list, dynamic_pos_list = [], [], [], [], []
    for start, end in selected_frames:
        all_indices = torch.arange(seq_len, device=image_feat.device).unsqueeze(0)
        if start == end:
            static_feat_list.append(torch.empty((0, embed_dim), device=image_feat.device))
            dynamic_feat_list.append(image_feat[start:end + 1])
            dynamic_attn_list.append(attn_weights[start:end + 1])
            static_pos_list.append(torch.empty((0, seq_len), device=image_feat.device))
            dynamic_pos_list.append(all_indices)
        else:
            windows_size = end - start + 1
            mask = torch.all(feature_sim[start:end, :] > tau, dim=0)
            static_feat_list.append(image_feat[start:end + 1, mask].mean(dim=0))
            dynamic_feat_list.append(image_feat[start:end + 1, ~mask])
            dynamic_attn_list.append(attn_weights[start:end + 1, ~mask])
            static_pos_list.append(all_indices[:, mask].expand(1, -1))
            dynamic_pos_list.append(all_indices[:, ~mask].expand(windows_size, -1))
    return static_feat_list, dynamic_feat_list, None, dynamic_attn_list, static_pos_list, dynamic_pos_list

@torch.no_grad()
def holitom_compress(X, attn, r, seed):
    """X: (T, S, D) pooled projected tokens; attn: (T, S) pooled last-vision-layer attention (mean heads, queries)."""
    tau = HOLITOM["T"][r]
    Beta, D, K, max_window_size = HOLITOM["beta"], HOLITOM["D"], HOLITOM["K"], HOLITOM["max_window"]
    gen = torch.Generator(device=X.device).manual_seed(seed)
    pooled_image_feat = X
    attn_weights = attn.to(X.device, X.dtype)
    batch_size, seq_len, embed_dim = pooled_image_feat.shape
    normed = torch.nn.functional.normalize(pooled_image_feat, p=2, dim=-1)
    feature_sim = torch.nn.functional.cosine_similarity(normed[:-1], normed[1:], dim=-1)
    selected_frames, total_reduced = ht_select_static_windows(feature_sim, batch_size, tau, max_window_size)
    total_tokens = batch_size * seq_len
    retain_ratio = min(r / ((total_tokens - total_reduced) / total_tokens), 1)
    sf, df, _, da, sp, dpos = ht_get_static_dynamic_features(pooled_image_feat, attn_weights, selected_frames,
                                                             feature_sim, tau)
    segs = []
    for i, (start, end) in enumerate(selected_frames):
        segs.append(ht_holitom(sf[i], df[i], da[i], sp[i], dpos[i], end - start + 1, retain_ratio, D, Beta, K,
                               X.dtype, gen))
    toks = torch.cat(segs, dim=0)
    inner = {"kind": "holitom", "layer": HOLITOM["inner_layer"], "merge_ratio": HOLITOM["inner_r"],
             "include_newline": True}
    return {"tokens": toks, "pos": np.arange(toks.shape[0]), "newline": True, "inner": inner}


# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# LLaVA-Scissor (Sun et al., 2025) — github.com/HumanMLLM/LLaVA-Scissor, llava/model/llava_arch_zip.py
# (UnionFind + approximate_components copied verbatim; spatial SCC per frame, temporal SCC, then every original
# token merged into its most similar component token). As in that code, the video path is forced to "flat":
# no newline token, tokens in component order, contiguous positions. Its paper sets the budget only through the
# similarity threshold tau; to compare methods at the SAME budget per video we search tau per video so that the
# output size is as close as possible to round(r * T * S) (documented deviation). Merging sums are done in fp32.
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
from collections import defaultdict

class UnionFind:
    def __init__(self, size):
        self.parent = np.arange(size, dtype=np.int64)
        self.rank = np.zeros(size, dtype=np.int32)
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def batch_union(self, x_arr, y_arr):
        for x, y in zip(x_arr, y_arr):
            x_root = self.find(x)
            y_root = self.find(y)
            if x_root == y_root:
                continue
            if self.rank[x_root] < self.rank[y_root]:
                self.parent[x_root] = y_root
            else:
                self.parent[y_root] = x_root
                if self.rank[x_root] == self.rank[y_root]:
                    self.rank[x_root] += 1

def approximate_components(adj_matrix, epsilon=0.05, rng=np.random):
    n = adj_matrix.shape[0]
    all_nodes = np.ones(n)
    all_indices = np.arange(0, n)
    if n == 0:
        return []
    sample_size = min(n, int(np.ceil(np.log(n) / epsilon ** 2)))
    sampled_nodes = rng.choice(n, size=sample_size, replace=False)
    all_nodes[sampled_nodes] = 0
    neighbor_dict = defaultdict(list)
    for i in sampled_nodes:
        neighbors = np.nonzero(adj_matrix[i])[0]
        valid_neighbors = np.intersect1d(neighbors, all_indices, assume_unique=True)
        neighbor_dict[i] = valid_neighbors
        all_nodes[neighbors] = 0
    remain_nodes = np.nonzero(all_nodes)[0]
    remain_nodes = [[element] for element in remain_nodes]
    uf = UnionFind(n)
    all_x, all_y = [], []
    for i in sampled_nodes:
        for j in neighbor_dict[i]:
            all_x.append(i)
            all_y.append(j)
    uf.batch_union(np.array(all_x), np.array(all_y))
    sampled_roots = np.array([uf.find(i) for i in sampled_nodes])
    unique_roots, counts = np.unique(sampled_roots, return_counts=True)
    components = []
    for root in unique_roots:
        cluster = np.where(uf.parent == root)[0].tolist()
        if len(cluster) > 0:
            components.append(cluster)
    components.extend(remain_nodes)
    degrees = np.count_nonzero(adj_matrix, axis=1)
    def get_sort_key(cluster):
        max_degree = -1
        min_node = float("inf")
        for node in cluster:
            current_degree = degrees[node]
            if (current_degree > max_degree) or (current_degree == max_degree and node < min_node):
                max_degree = current_degree
                min_node = node
        return min_node
    components.sort(key=get_sort_key)
    return components

def _scissor_once(Xf, tau, eps, rng):
    """Xf: (T, S, D) float32. Returns (selected tokens float32, count). Same steps as the official code."""
    T, S, D = Xf.shape
    nrm = Xf / torch.norm(Xf, p=2, dim=-1, keepdim=True)
    high = (torch.matmul(nrm, nrm.transpose(1, 2)) > tau).cpu().numpy()
    fused = []
    for f in range(T):
        for comp in approximate_components(high[f], epsilon=eps, rng=rng):
            fused.append(Xf[f, comp].mean(dim=0))
    sel = torch.stack(fused)
    sn = sel / torch.norm(sel, p=2, dim=1, keepdim=True)
    high2 = (sn @ sn.t() > tau).cpu().numpy()
    sel = torch.stack([sel[comp].mean(dim=0) for comp in approximate_components(high2, epsilon=eps, rng=rng)])
    return sel

def _scc_count_fast(Xf, tau):
    """Exact connected-component count with scipy (used only to choose tau quickly)."""
    from scipy.sparse.csgraph import connected_components
    from scipy.sparse import csr_matrix
    T, S, D = Xf.shape
    nrm = Xf / torch.norm(Xf, p=2, dim=-1, keepdim=True)
    high = (torch.matmul(nrm, nrm.transpose(1, 2)) > tau).cpu().numpy()
    means = []
    for f in range(T):
        k, lab = connected_components(csr_matrix(high[f]), directed=False)
        lab_t = torch.as_tensor(lab, device=Xf.device)
        m = torch.zeros(k, D, device=Xf.device).index_add_(0, lab_t, Xf[f])
        means.append(m / torch.bincount(lab_t, minlength=k).unsqueeze(1).float())
    sel = torch.cat(means)
    sn = sel / torch.norm(sel, p=2, dim=1, keepdim=True)
    k2, _ = connected_components(csr_matrix((sn @ sn.t() > tau).cpu().numpy()), directed=False)
    return k2

@torch.no_grad()
def scissor_compress(X, r, seed):
    T, S, D = X.shape
    Xf = X.float()
    target = max(1, round(r * T * S))
    eps = SCISSOR["epsilon"]
    # 1) quick bracket with an exact connected-component count (scipy)
    lo, hi = -1.0, 1.0
    for _ in range(SCISSOR["search_steps"]):
        mid = (lo + hi) / 2
        if _scc_count_fast(Xf, mid) < target:
            lo = mid
        else:
            hi = mid
    tau0 = hi
    # 2) refine with the official (sampled, approximate) algorithm itself, same seed at every tau
    cache = {}
    def run(t):
        t = round(t, 6)
        if t not in cache:
            cache[t] = _scissor_once(Xf, t, eps, np.random.RandomState(seed))
        return cache[t]
    lo, hi = max(-1.0, tau0 - 0.1), min(1.0, tau0 + 0.02)
    while run(lo).shape[0] > target and lo > -1.0:
        lo = max(-1.0, lo - 0.1)
    while run(hi).shape[0] < target and hi < 1.0:
        hi = min(1.0, hi + 0.05)
    for _ in range(SCISSOR["refine_steps"]):
        mid = (lo + hi) / 2
        if run(mid).shape[0] < target:
            lo = mid
        else:
            hi = mid
    best = min([lo, hi], key=lambda t: abs(run(t).shape[0] - target))
    sel = run(best)
    # merge every original token into its most similar selected token (official final step)
    rem = Xf.reshape(-1, D)
    simm = (rem / torch.norm(rem, p=2, dim=1, keepdim=True)) @ (sel / torch.norm(sel, p=2, dim=1, keepdim=True)).t()
    closest = torch.argmax(simm, dim=1)
    merged = torch.zeros_like(sel).scatter_add_(0, closest.view(-1, 1).expand(-1, D), rem)
    counts = torch.bincount(closest, minlength=sel.shape[0]).float() + 1
    toks = ((sel + merged) / counts.unsqueeze(1)).to(X.dtype)
    return {"tokens": toks, "pos": np.arange(toks.shape[0]), "newline": False, "inner": None, "tau": best}



def _contig(t):
    return {"tokens": t, "pos": np.arange(t.shape[0]), "n_full": int(t.shape[0]), "newline": True, "inner": None}

def compress_all(X, A, G, VA, VM, seed, only=None, skip=()):
    """Every (method, setting) -> dict(tokens, pos, n_full, newline, inner). Official code is kept as is; if it fails
    on a video, that (method, setting) is skipped for the video and reported — never silently replaced."""
    T, L, D = X.shape
    jobs = []
    for r in METHOD_GRID["flashvid"]:
        jobs.append(("flashvid", str(r), lambda r=r: flashvid_compress(X, VA, r)))
    for r in METHOD_GRID["fastvid"]:
        def fv(r=r):
            t, i = fastvid_compress(X, A, G, r, seed=seed, **FASTVID)
            return {"tokens": t, "pos": i.cpu().numpy(), "n_full": T * L, "newline": True, "inner": None}
        jobs.append(("fastvid", str(r), fv))
    for r in METHOD_GRID["holitom"]:
        jobs.append(("holitom", str(r), lambda r=r: holitom_compress(X, VA, r, seed)))
    for r in METHOD_GRID["visionzip"]:
        jobs.append(("visionzip", str(r), lambda r=r: _contig(visionzip_compress(X, VA, VM, r))))
    for r in METHOD_GRID["scissor"]:
        jobs.append(("scissor", str(r), lambda r=r: scissor_compress(X, r, seed)))
    for K in METHOD_GRID["dycoke"]:
        jobs.append(("dycoke", f"K{K}", lambda K=K: _contig(dycoke_ttm(X, K))))
    for r in METHOD_GRID["uniform"]:
        jobs.append(("uniform", str(r), lambda r=r: _contig(uniform_pool(X, r))))
    for r in METHOD_GRID["random"]:
        jobs.append(("random", str(r), lambda r=r: _contig(random_drop(X, r, seed))))
    out, failures = {}, []
    for m, s, fn in jobs:
        if (only is not None and m not in only) or m in skip:
            continue
        try:
            c = fn()
            if c.get("n_full") is None or m in ("flashvid", "holitom", "scissor"):
                c["n_full"] = int(c["tokens"].shape[0])
            out[(m, s)] = c
        except Exception as e:
            failures.append((m, s, f"{type(e).__name__}: {e}"))
    return out, failures

# ── self-test on smooth random "video" features (shapes, finiteness, budgets) ──
def _fake_video(T, S, D, dev, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    base = torch.randn(1, S, D, generator=g)
    drift = torch.cumsum(0.25 * torch.randn(T, 1, D, generator=g), 0)
    return (base + drift + 0.6 * torch.randn(T, S, D, generator=g)).to(dev, torch.float16)

_T, _L = NUM_FRAMES, POOLED * POOLED
_X = _fake_video(_T, _L, 64, EMB_DEVICE)
_A = torch.rand(_T, _L, device=EMB_DEVICE)
_G = torch.randn(_T, 32, device=EMB_DEVICE)
_VA = torch.rand(_T, _L, device=EMB_DEVICE); _VA = _VA / _VA.sum(1, keepdim=True)
_VM = torch.randn(_T, _L, 16, device=EMB_DEVICE)
_comp, _fails = compress_all(_X, _A, _G, _VA, _VM, seed=0)
assert not _fails, _fails
assert len(_comp) == sum(len(v) for v in METHOD_GRID.values()), "missing compressed passes"
for (m, s), c in _comp.items():
    t = c["tokens"]
    assert torch.isfinite(t.float()).all(), (m, s)
    assert len(c["pos"]) == t.shape[0] and int(np.max(c["pos"])) < c["n_full"]
    print(f"  {m:9s} {s:5s} enters LLM with {t.shape[0]:5d} tokens = {t.shape[0] / (_T * _L):.3f}"
          + (f" | inner {c['inner']['kind']} @ layer {c['inner']['layer']}" if c["inner"] else ""))
del _X, _A, _G, _VA, _VM, _comp
print(f"Compressor self-test OK ✅  ({sum(len(v) for v in METHOD_GRID.values())} compressed passes)")

### 6 · Main run

In [ ]:
# ── Main run: per video encode 3 frame sets, compress once, cache each prefix once, run every question on it ──
import zlib
done = set()
if os.path.exists(RES_PATH):
    with open(RES_PATH) as f:
        for line in f:
            try:
                done.add(json.loads(line)["qid"])
            except Exception:
                pass
print(f"{len(done)} questions already in results (resume)")

table = build_manifest()
T_START = time.time()
time_left = lambda: MAX_RUNTIME_HOURS * 3600 - (time.time() - T_START)

def video_groups():
    """Yields (video_key, local path, DataFrame of its questions still to run)."""
    if BENCHMARK == "videomme":
        for ch in VIDEOMME_CONFIRM_CHUNKS:
            if time_left() < 1800:
                print("⏱️ Time budget reached — skipping remaining chunks"); return
            vids = videomme_fetch_chunk(ch)
            sub = table[(table.video_key.isin(vids.keys()) | table.video_alt.isin(vids.keys())) & ~table.qid.isin(done)]
            if QUICK_MAX_Q:
                sub = sub[sub.video_key.isin(sub.video_key.drop_duplicates().head(max(1, QUICK_MAX_Q // 3)))]
            print(f"Chunk {ch}: {len(sub)} questions, {sub.video_key.nunique()} videos")
            for vk, g in sub.groupby("video_key", sort=False):
                yield vk, vids.get(vk) or vids.get(g.video_alt.iloc[0]), g
            shutil.rmtree(os.path.join(WORK_DIR, f"vmme_{ch}"), ignore_errors=True)
    else:
        sub = table[~table.qid.isin(done)]
        if QUICK_MAX_Q:
            per = max(1, QUICK_MAX_Q // max(1, sub.group.nunique()))
            sub = sub.groupby("group", group_keys=False).head(per).head(QUICK_MAX_Q)
        keys = sorted(sub.video_key.unique())
        print(f"{len(sub)} questions on {len(keys)} videos to run; downloading videos…")
        fetch = {"mvbench": mvbench_fetch, "egoschema": egoschema_fetch,
                 "longvideobench": longvideobench_fetch, "mlvu": mlvu_fetch}[BENCHMARK]
        if BENCHMARK == "mlvu":
            # MLVU videos are large (~70-80 GB for the sample, more than Kaggle's disk): download 8 at a time
            # and delete each batch once no video in it can still be in use.
            yield from _batched_groups(sub, keys, fetch, batch=8)
            return
        paths = fetch(keys)
        missing = [k for k in keys if k not in paths]
        for k in missing:
            with open(ERR_PATH, "a") as f:
                f.write(f"{k}\tmissing video\t{len(sub[sub.video_key == k])} questions dropped\n")
        if missing:
            print(f"⚠️ {len(missing)} videos could not be found/downloaded (listed in errors.log)")
        for vk, g in sub[sub.video_key.isin(paths.keys())].groupby("video_key", sort=False):
            yield vk, paths[vk], g

def _remove_files(paths):
    for p in paths:
        try:
            os.remove(p)
        except OSError:
            pass
    _clean_cache()

def _batched_groups(sub, keys, fetch, batch):
    prev, prev2 = [], []
    for i in range(0, len(keys), batch):
        if time_left() < 1800:
            print("⏱️ Time budget reached — not downloading more videos"); break
        ks = keys[i:i + batch]
        # When we get here, the main loop is still reading the last video of the previous batch,
        # so only the batch before that one is safe to delete.
        _remove_files(prev2)
        paths = fetch(ks)
        for k in ks:
            if k not in paths:
                with open(ERR_PATH, "a") as f:
                    f.write(f"{k}\tmissing video\t{len(sub[sub.video_key == k])} questions dropped\n")
        print(f"  batch {i // batch + 1}/{-(-len(keys) // batch)}: {len(paths)}/{len(ks)} videos | "
              f"free disk {shutil.disk_usage(WORK_DIR).free / 1e9:.0f} GB")
        for k in ks:
            if k in paths:
                yield k, paths[k], sub[sub.video_key == k]
        prev2, prev = prev, list(paths.values())

def _finish(res, row, T, L):
    res["correct"] = bool(res["pred"] == row.answer)
    return res

def run_video(Xf, A, G, VA, VM, Xshift, rows, seed, first):
    T, L, D = Xf.shape
    pool = cf.ThreadPoolExecutor(1)
    scissor_job = pool.submit(compress_all, Xf, A, G, VA, VM, seed, ("scissor",))       # CPU-heavy: runs alongside
    comp, fails = compress_all(Xf, A, G, VA, VM, seed, skip=("scissor",))
    if first:
        cache_check(Xf, comp, rows)
    recs = {}
    for row in rows:
        recs[row.qid] = {"qid": row.qid, "video": row.video_key, "group": row.group, "task": row.task,
                         "gt": row.answer, "n_opt": min(len(row.options), len(LETTERS)), "n_full": T * L,
                         "noise": {}, "methods": {}}
    sufs = {row.qid: build_suffix_ids(row.question, row.options) for row in rows}
    # full model + option-order noise share one prefix
    P = prefill(Xf.reshape(-1, D), np.arange(T * L), T * L)
    for row in rows:
        n = recs[row.qid]["n_opt"]
        recs[row.qid]["full"] = _finish(suffix_pass(P, sufs[row.qid], n), row, T, L)
        for kind in NOISE_OPTION_ORDERS:
            res = suffix_pass(P, build_suffix_ids(row.question, reorder_options(row.options, kind)), n)
            recs[row.qid]["noise"][kind] = _finish(map_back(res, n, kind), row, T, L)
    del P
    # frame-shift noise
    for kind, Xk in Xshift.items():
        P = prefill(Xk.reshape(-1, D), np.arange(T * L), T * L)
        for row in rows:
            recs[row.qid]["noise"][kind] = _finish(suffix_pass(P, sufs[row.qid], recs[row.qid]["n_opt"]), row, T, L)
        del P
    def run_comp(comp):
        for (m, s), c in comp.items():
            if c["inner"] is None:          # question-independent: cache the prefix
                P = prefill(c["tokens"], c["pos"], c["n_full"], newline=c["newline"])
                outs = {row.qid: suffix_pass(P, sufs[row.qid], recs[row.qid]["n_opt"]) for row in rows}
                del P
            else:                           # FlashVID / HoliTom act inside the LLM and depend on the question
                outs = {row.qid: llm_pass(c["tokens"], c["pos"], c["n_full"], sufs[row.qid], recs[row.qid]["n_opt"],
                                          newline=c["newline"], inner=c["inner"]) for row in rows}
            for row in rows:
                res = _finish(outs[row.qid], row, T, L)
                res["retention"] = round(c["tokens"].shape[0] / (T * L), 4)
                res["retention_layer_avg"] = round(res["vis_mean_layers"] / (T * L), 4)
                if "tau" in c:
                    res["tau"] = round(float(c["tau"]), 5)
                recs[row.qid]["methods"].setdefault(m, {})[s] = res
    run_comp(comp)
    comp2, fails2 = scissor_job.result()
    pool.shutdown()
    run_comp(comp2)
    return [recs[row.qid] for row in rows], fails + fails2

n_new, n_err, consec_fail, FIRST = 0, 0, 0, True
def _fail_guard():
    if consec_fail >= 6:
        raise RuntimeError("6 questions failed in a row — stopping instead of wasting GPU hours. Send me the log.")

reader = cf.ThreadPoolExecutor(1)
groups = video_groups()
nxt = next(groups, None)
fut = reader.submit(read_all_frame_sets, nxt[1]) if nxt else None
while nxt is not None:
    vk, path, g = nxt
    cur_fut = fut
    nxt = next(groups, None) if time_left() > 1200 else None
    if nxt is None and time_left() <= 1200:
        print("⏱️ Time budget reached — stopping after this video")
    fut = reader.submit(read_all_frame_sets, nxt[1]) if nxt else None       # decode the next video meanwhile
    t0 = time.time()
    rows = list(g.itertuples())
    try:
        fsets = cur_fut.result()
        if FIRST:
            equivalence_check(fsets["orig"], rows[0])
        Xf, A, G, VA, VM = encode_video(preprocess(fsets["orig"]))
        Xshift = {k: encode_video(preprocess(fsets[k]))[0] for k in NOISE_FRAME_SHIFTS}
        recs, fails = run_video(Xf, A, G, VA, VM, Xshift, rows, zlib.crc32(str(vk).encode()), FIRST)
        FIRST = False
    except RuntimeError as e:
        if "check failed" in str(e) or "Equivalence check" in str(e) or "Cache check" in str(e):
            raise
        n_err += len(rows); consec_fail += len(rows)
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tvideo\t{type(e).__name__}: {e}\n")
        print(f"  ✗ video {vk}: {type(e).__name__}: {e}"); _fail_guard(); continue
    except Exception as e:
        n_err += len(rows); consec_fail += len(rows)
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tvideo\t{type(e).__name__}: {e}\n")
        print(f"  ✗ video {vk}: {type(e).__name__}: {e}"); torch.cuda.empty_cache(); _fail_guard(); continue
    for m, s_, err in fails:
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tcompress {m}@{s_}\t{err}\n")
        print(f"  ! {m}@{s_} failed on video {vk} (skipped for this video): {err}")
    with open(RES_PATH, "a") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
            done.add(r["qid"])
    n_new += len(recs); consec_fail = 0
    for r in recs:
        flips = sum(x["pred"] != r["full"]["pred"] for v in r["methods"].values() for x in v.values())
        nz = "".join("·" if x["pred"] == r["full"]["pred"] else "x" for x in r["noise"].values())
        print(f"  ✓ {r['qid']} [{r['group']}] gt={r['gt']} full={r['full']['pred']} noise[{nz}] "
              f"| {flips}/{sum(len(v) for v in r['methods'].values())} compressed changed")
    print(f"    video done in {time.time() - t0:.0f}s | {len(done)} questions total | {time_left() / 3600:.1f} h left")
    del Xf, A, G, VA, VM, Xshift, fsets
    gc.collect(); torch.cuda.empty_cache()
reader.shutdown()
print(f"\nFinished: {n_new} new questions, {n_err} errors, {len(done)} total in {RES_PATH}")

### 7 · Per-benchmark analysis

In [ ]:
# ── Per-benchmark analysis (pre-registered measures; the cross-benchmark hypothesis tests are run afterwards) ──
import matplotlib.pyplot as plt
R = []
with open(RES_PATH) as f:
    for line in f:
        try:
            R.append(json.loads(line))
        except Exception:
            pass
R = [r for r in R if len(r.get("noise", {})) == 4]
n = len(R)
if n < 5:
    raise SystemExit(f"Only {n} results — nothing to analyse yet (normal in a quick test with very few questions).")
rng = np.random.default_rng(SEED)
vid = np.array([r["video"] for r in R]); videos = np.unique(vid)
v_rows = [np.flatnonzero(vid == v) for v in videos]
B = 2000
boot = [np.concatenate([v_rows[j] for j in rng.integers(0, len(videos), len(videos))]) for _ in range(B)]
full = np.array([r["full"]["pred"] for r in R]); fcor = np.array([r["full"]["correct"] for r in R])
margin = np.array([r["full"]["margin"] for r in R])
NK = ["frame_shift", "frame_shift_q", "option_cycle", "option_reverse"]
NF = {k: np.array([r["noise"][k]["pred"] for r in R]) != full for k in NK}
STABLE = {k: sum(NF[j] for j in NK if j != k) == 0 for k in NK}       # stable under the other three
DELTA = 0.03

def kl(p_lp, q_lp):
    p_lp, q_lp = np.array(p_lp), np.array(q_lp)
    return float((np.exp(p_lp) * (p_lp - q_lp)).sum())

def pass_arrays(get):
    ok = np.array([get(r) is not None for r in R])
    pred = np.array([get(r)["pred"] if o else "" for r, o in zip(R, ok)])
    cor = np.array([bool(get(r)["correct"]) if o else False for r, o in zip(R, ok)])
    return ok, pred, cor

def excess(flip, ok, idx):
    """Pre-registered Δ: mean over the 4 held-out noise passes of [method flips − held-out noise flips] on the
    questions stable under the other three noise passes (restricted to questions where the method ran)."""
    vals = []
    for k in NK:
        s = STABLE[k][idx] & ok[idx]
        if s.sum() == 0:
            return np.nan
        vals.append(flip[idx][s].mean() - NF[k][idx][s].mean())
    return float(np.mean(vals))

def describe(get, name):
    ok, pred, cor = pass_arrays(get)
    flip = (pred != full) & ok
    idx_all = np.arange(n)
    d = excess(flip, ok, idx_all)
    d_b = np.array([excess(flip, ok, b) for b in boot])
    lo, hi = np.nanpercentile(d_b, [2.5, 97.5])
    h, g = int((fcor & ~cor & ok).sum()), int((~fcor & cor & ok).sum())
    rel_b = [100 * cor[b][ok[b]].mean() / max(fcor[b][ok[b]].mean(), 1e-9) for b in boot]
    return {"pass": name, "coverage": float(ok.mean()), "acc": float(cor[ok].mean()),
            "rel_acc": float(100 * cor[ok].mean() / fcor[ok].mean()),
            "rel_ci": [float(np.percentile(rel_b, 2.5)), float(np.percentile(rel_b, 97.5))],
            "flip": float(flip[ok].mean()), "excess": d, "excess_ci": [float(lo), float(hi)],
            "class": "lossy" if lo > 0 else ("noise-equivalent" if hi < DELTA else "inconclusive"),
            "harmful": h, "helpful": g, "harmful_share": h / (h + g) if h + g else float("nan"),
            "kl": float(np.mean([kl(R[k]["full"]["logp"], get(R[k])["logp"]) for k in np.flatnonzero(ok)])),
            "flops_ratio": float(np.mean([get(R[k])["flops"] for k in np.flatnonzero(ok)]) /
                                 np.mean([R[k]["full"]["flops"] for k in np.flatnonzero(ok)])),
            "retention": float(np.mean([get(R[k]).get("retention", 1.0) for k in np.flatnonzero(ok)])),
            "retention_layer_avg": float(np.mean([get(R[k]).get("retention_layer_avg", get(R[k]).get("retention", 1.0))
                                                 for k in np.flatnonzero(ok)]))}

# ---------------- noise ----------------
fragility = float(np.any(np.stack([NF[k] for k in NK]), 0).mean())
fr_b = [np.any(np.stack([NF[k][b] for k in NK]), 0).mean() for b in boot]
noise_rows = []
for k in NK:
    ok, pred, cor = pass_arrays(lambda r, k=k: r["noise"][k])
    h, g = int((fcor & ~cor).sum()), int((~fcor & cor).sum())
    noise_rows.append({"pass": k, "flip": float(NF[k].mean()), "acc": float(cor.mean()),
                       "heldout_flip_on_stable": float(NF[k][STABLE[k]].mean()), "harmful": h, "helpful": g})
accs = [fcor.mean()] + [x["acc"] for x in noise_rows]
noise_range = 100 * (max(accs) - min(accs))
H_n = sum(x["harmful"] for x in noise_rows); G_n = sum(x["helpful"] for x in noise_rows)
print(f"=== {BENCHMARK}: {n} questions, {len(videos)} videos | full accuracy {fcor.mean():.3f} ===")
print(f"Fragility (changed by ≥1 of 4 noise passes): {fragility:.3f} "
      f"(95% CI {np.percentile(fr_b, 2.5):.3f}–{np.percentile(fr_b, 97.5):.3f})")
print(f"Noise accuracy range: {noise_range:.1f} points | pooled noise harmful share {H_n / max(H_n + G_n, 1):.2f}")
for x in noise_rows:
    print(f"   {x['pass']:15s} flips {x['flip']:.3f} | held-out flips on stable questions {x['heldout_flip_on_stable']:.3f}")

# ---------------- methods ----------------
rows = []
for m, settings in METHOD_GRID.items():
    for s in settings:
        key = f"K{s}" if m == "dycoke" else str(s)
        rows.append({"method": m, "setting": key,
                     **describe(lambda r, m=m, key=key: r["methods"].get(m, {}).get(key), f"{m}@{key}")})
cells = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
show = cells[["method", "setting", "coverage", "retention", "rel_acc", "flip", "excess", "excess_ci", "class",
              "harmful_share", "kl", "flops_ratio"]].copy()
show["excess_ci"] = show.excess_ci.apply(lambda c: f"{c[0]:+.3f}…{c[1]:+.3f}")
print("\nCells (excess = extra answer changes beyond held-out noise, on noise-stable questions):")
print(show.round(3).to_string(index=False))
cells.to_csv(os.path.join(OUT_DIR, "cells.csv"), index=False)

# ---------------- confidence quartiles ----------------
q = np.digitize(margin, np.quantile(margin, [0.25, 0.5, 0.75]))
noise_any = np.mean(np.stack([NF[k] for k in NK]), 0)
low_keys = [(m, (f"K{s}" if m == "dycoke" else str(s))) for m, v in METHOD_GRID.items() for s in v
            if m != "dycoke" and s <= 0.10]
cf_low = np.mean(np.stack([np.array([(r["methods"].get(m, {}).get(s) or r["full"])["pred"] != r["full"]["pred"]
                                     for r in R]) for m, s in low_keys]), 0)
quart = {"noise": [float(noise_any[q == i].mean()) for i in range(4)],
         "compressed_le10": [float(cf_low[q == i].mean()) for i in range(4)]}
print("\nFlip rate by full-model confidence quartile (Q1 = least sure):", {k: np.round(v, 3).tolist() for k, v in quart.items()})

summary = {"benchmark": BENCHMARK, "n": n, "n_videos": int(len(videos)), "model": MODEL_ID, "frames": NUM_FRAMES,
           "full_accuracy": float(fcor.mean()), "fragility": fragility,
           "fragility_ci": [float(np.percentile(fr_b, 2.5)), float(np.percentile(fr_b, 97.5))],
           "noise_accuracy_range_points": noise_range, "noise": noise_rows,
           "noise_harmful_share": H_n / max(H_n + G_n, 1), "confidence_quartiles": quart,
           "cells": rows, "time_hours": (time.time() - T_START) / 3600 if "T_START" in dir() else None}
with open(os.path.join(OUT_DIR, f"summary_{BENCHMARK}.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
# ---------------- plot (after saving the numbers, so a plotting problem can never lose them) ----------------
try:
    col = {"flashvid": "#17becf", "fastvid": "#1f77b4", "holitom": "#9467bd", "visionzip": "#d62728",
           "scissor": "#8c564b", "dycoke": "#2ca02c", "uniform": "#7f7f7f", "random": "#ff7f0e"}
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    for m, g in cells.groupby("method", sort=False):
        g = g.sort_values("retention_layer_avg")
        x = 100 * g.retention_layer_avg.values
        ax[0].errorbar(x, g.excess, yerr=[g.excess - g.excess_ci.str[0], g.excess_ci.str[1] - g.excess],
                       fmt="o-", color=col[m], label=m, capsize=2, ms=4)
        ax[1].plot(x, g.rel_acc, "o-", color=col[m], label=m)
    ax[0].axhline(0, c="k", lw=1); ax[0].axhline(DELTA, c="k", lw=0.8, ls=":")
    ax[0].set_xscale("log"); ax[0].set_xlabel("visual tokens kept (%)"); ax[0].set_ylabel("excess flips beyond noise")
    ax[0].set_title(f"{BENCHMARK}: extra answer changes beyond harmless noise"); ax[0].legend(fontsize=8, ncol=2)
    ax[1].axhline(100, c="k", lw=0.8, ls="--"); ax[1].set_xscale("log")
    ax[1].set_xlabel("visual tokens kept (%)"); ax[1].set_ylabel("relative accuracy (%)"); ax[1].set_title("Relative accuracy")
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"summary_{BENCHMARK}.png"), dpi=150); plt.show()
except Exception as e:
    plt.close("all")
    print(f"(plot skipped: {type(e).__name__}: {e})")

if QUICK_TEST:
    print("\n(Quick test: numbers are meaningless — this only proves the pipeline works.)")
print("Files saved in", OUT_DIR)

## Next
Send the files listed at the top. After all five benchmarks are in, the pre-registered hypotheses (H1–H5) are tested across benchmarks in a separate analysis.